## Tutorial: Accessing ADT-HURSAT Data ##

In this tutorial, you will learn how to:

* Efficiently retrieve ADT-HURSAT data from the Google Bucket
* Visualize the ADT-HURSAT data
* Compare the ADT-HURSAT data with the IBTrACS data set

This notebook utilizes the ADT-HURSAT dataset to analyze tropical cyclone storm counts and intensities within specific ocean basins. The ADT-HURSAT dataset provides climate-quality tropical cyclone intensity estimates from 1978 to 2024, derived from applying the Advanced Dvorak Technique (ADT) to Hurricane Satellite (HURSAT) data.

Furthermore, this notebook facilitates a comparison between ADT-HURSAT data and the International Best Track Archive for Climate Stewardship [IBTrACS](https://www.ncei.noaa.gov/products/international-best-track-archive) data, the most comprehensive global collection of tropical cyclones available, to evaluate how these datasets represent such phenomena. The [ADT-HURSAT product]((https://www.ncei.noaa.gov/products/advanced-dvorak-technique-hurricane-satellite)) is available through the National Center for Environmental Information, which also provides direct data access via the [Google Cloud bucket](https://console.cloud.google.com/storage/browser/noaa-ncei-ipg/datasets/hursat/adt/).

This notebook contains multiple sections that guide you through how to access the data, process it, and create visualizations. The Table of Contents contains detailed headers to direct you to the different parts of the notebook. The cells are also numbered to match the sections in the Table of Contents to make it easier to navigate through the notebook. **It is recommended that you read through the entire notebook before running it.** If you want to make changes to the notebook, make sure to save a copy.

In order for this notebook to work correctly, **all of the cells in Sections 1, 2, and 3 must be run.** Section 1 contains the installation and import of the python packages that are used in the notebook. Section 2 contains multiple subsections with custom functions to access, filter, and process the ADT-HURSAT and IBTrACS data. If you modify the functions in this section, the notebook may not work as expected. Section 3 contains the calls to the custom functions to read in, process, and analyze the data. Section 3.4 specifies the basin that you would like to use for all of the data visualizations and comparison with the IBTrACS data. If you would like to switch basins, you can modify the basin_to_filter variable, save your changes, and rerun the cell. Section 3 also conducts the data processing needed to generate the figures. Sections 4 and 5 contain multiple cells to generate the data visualizations. Each cell contains a dictionary of the plotting parameters that you can modify to style the figure as you see fit. Alternative text is automatically generated for the data visualizations; however, missing data or modifications that you make to the figure may prevent the alternative text from being generated. Scientific interpretations for the figures are provided as figure captions, including the use of hurricane terminology for classifications, assuming you choose the North Atlantic basin. The specific information in the interpretations may not match other basins, but you can use the information for the North Atlantic basin as a starting point to interpret what the visualizations are displaying for other basins. 
The analyses in this notebook use the nearest storm fixes, which are data that are provided at 6 hourly intervals (0, 6, 12, and 18 UTC), to match initial source agency best track data. Intensity information is measured as the maximum 1-minute sustained wind speed of a storm (USA_wind variable) in IBTrACS and a similar metric determined by the ADT-algorithm (WindSpeed variable). The words intensity and wind speed are used interchangeably throughout the notebook. 

This notebook does not provide code to download the ADT-HURSAT data from the Google Cloud Bucket. You can use some of the filter functions in Section 6 to identify the files that you are interested in downloading, and then develop your own code or manually download those files from the Google Cloud Bucket. Note that testing for statistical significance is not included in the notebook. For more information about this product, including file naming convention, visit the [ADT-HURSAT product page](https://www.ncei.noaa.gov/products/advanced-dvorak-technique-hurricane-satellite). This notebook was created as part of an initiative to develop industry-tailored resources. For more information, visit the [Our Impact](https://www.ncei.noaa.gov/about/our-impact) website. For questions about this notebook, contact industryproving.grounds@noaa.gov. 


## --1-- Import statements

This section sets up the analysis environment by importing the necessary Python libraries.
* **Data Handling:** `pandas` and `xarray` are used for structured data manipulation.
* **Visualization:** `matplotlib` and `seaborn` provide plotting capabilities.
* **Data Access:** `google.cloud.storage` allows connection to the NOAA Google Cloud Bucket.
* **Utilities:** Libraries like `io`, `os`, and `concurrent.futures` handle file operations and parallel processing.

In [ ]:
'''
 0 Dependency Installation
'''

# If you are running this notebook in a new environment (e.g., Google Colab,
# local virtualenv) you will need to install the necessary libraries.
# If you are running the notebook multiple times in the same environment,
# You can comment out the line below. The install only needs to happen once.

%pip install xarray fsspec netcdf4 requests google-cloud-storage pillow seaborn

In [ ]:
"""
1 Import Statements
"""

# Standard Library
import os # Used for interacting with the operating system
import sys # Provides access to system-specific parameters and functions
import io # Core tools for working with streams of various types
import re # Regular expression operations
import calendar
import textwrap
import traceback # Module to extract, format and print stack traces
from datetime import datetime, timedelta # Classes for manipulating dates and times
from concurrent.futures import ThreadPoolExecutor, as_completed # For running functions in separate threads

# Third-Party Data & Computation
import numpy as np # Scientific computing library
import pandas as pd # Data manipulation and analysis library
import xarray as xr # Labeled multi-dimensional arrays
import fsspec # Provides a unified interface to local, remote and embedded file systems
from scipy.stats import linregress # Calculates a linear least-squares regression for two sets of measurements

# Visualization
import seaborn as sns # Statistical data visualization library
import matplotlib.pyplot as plt # Plotting library
import matplotlib.ticker as ticker # Contains classes for controlling tick locating and formatting
import matplotlib.dates as mdates # Provides tools for handling dates on matplotlib axes
from PIL import Image # Python Imaging Library (PIL) for image processing

# Network & Cloud
import requests # Library for making HTTP requests
from google.cloud import storage # Client library for Google Cloud Storage
import glob # Finds pathnames matching a specified pattern


print("Imports complete.")

## --2-- Custom functions and utilities

This section defines the specific tools required to analyze the tropical cyclone data. It is organized into five modular subsections for better maintainability:
 - **Data Access, Parsing, and Validation (2.1):** Functions to read raw IBTrACS data, parse NetCDF files, and validate user inputs like year and month.
 - **ADT-HURSAT File and Storm Identification (2.2):** Utilities to list files from the Google Cloud bucket, determine storm basins using boundary images, and filter file lists.
 - **Core Data Transformation and Merging (2.3):** The primary logic for standardizing time steps (rounding to 3H), calculating intensity thresholds (Major/Hurricane/Named), and merging the ADT-HURSAT and IBTrACS datasets.
 - **Accessibility Helpers (2.4):** Helper functions to format text for screen readers (e.g., spacing out acronyms) and wrap text for cleaner output.
 - **Visualization Generation (2.5):** Custom plotting functions that create Bar Charts, Time Series, and Histograms, complete with automatically generated, detailed Alt Text for accessibility compliance.

### --2.1-- Data Access, Parsing, and Validation Functions

These functions deal with fetching external IBTrACS data and ensuring user input is clean and usable, enforcing Robust Input Validation.

**Note:** The boundary image loaded here is used specifically to subdivide the **North Atlantic (NA)** and **Eastern North Pacific (EP)** basins, resolving ambiguities in storm location near the basin boundaries.

In [ ]:
"""
2.1 Data Access, Parsing, and Validation Utilities
"""

def read_ibtracs_data_from_url(url):
  """
  Reads IBTrACS data from a netCDF file, extracts variables, decodes strings,
  renames columns, and filters the data for years between 1978 and 2024.

  Returns:
      pandas.DataFrame: A filtered DataFrame with the processed IBTrACS data.
                        Returns None if an error occurs.
  """
  try:
    #this fsspec section is a better way to handle the nc data from the public url
    with fsspec.open(url, "rb") as f:
        ds = xr.open_dataset(f)
        print("Dataset loaded successfully!")
        variables_to_extract = ds[['usa_lat','usa_lon','time', 'numobs', 'sid', 'season',
                                   'basin', 'name', 'iso_time', 'usa_wind',
                                   'usa_pres', 'usa_rmw', 'usa_eye']]
        # Convert the selected variables to a pandas DataFrame
        ibtracs_df = variables_to_extract.to_dataframe()

    for col in ['sid', 'basin', 'name', 'iso_time']:
        if col in ibtracs_df.columns:
            ibtracs_df[col] = ibtracs_df[col].str.decode('utf-8', errors='replace')

    ibtracs_df.rename(columns={'usa_lat': 'LAT_degrees_north',
                               'usa_lon': 'LON_degrees_east',
                               'sid': 'SID',
                               'season': 'SEASON_Year',
                               'basin': 'BASIN',
                               'name': 'NAME',
                               'iso_time': 'ISO_TIME',
                               'usa_wind': 'USA_WIND_kts',
                               'usa_pres': 'USA_PRES_mb',
                               'usa_rmw': 'USA_RMW_nmile',
                               'usa_eye': 'USA_EYE_nmile',
                               }, inplace=True)

    ibtracs_filtered_df = ibtracs_df[
        (ibtracs_df['SEASON_Year'] >= 1978) &
        (ibtracs_df['SEASON_Year'] <= 2024)
    ]

    return ibtracs_filtered_df

  except Exception as e:
    print(f"An error occurred while reading IBTrACS data: {e}")
    print("If you encounter an error here, please verify the IBTrACS data URL is accessible and try running this cell again.")
    traceback.print_exc()
    return None


def read_ibtracs_names_file(input_file):
  """
  Reads a CSV file containing IBTrACS names and supporting information from a
  Google Cloud Storage bucket, splits a column into 'Ibtracs_names' and 'Sources',
  and renames specific columns.

  Returns:
      pandas.DataFrame: A DataFrame containing the processed IBTrACS names and
                        supporting information. Returns None in case of an error.
  """
  try:
      ibtracs_names_df = pd.read_csv(input_file)

      # Split 'Names and sources in IBTrACS' column by '[' delimiter into two new columns
      ibtracs_names_df[['Ibtracs_names', 'Sources']] = \
                        ibtracs_names_df['Names and sources in IBTrACS'].str.split('[', expand=True,n=1)

      # Rename columns for clarity and consistency
      ibtracs_names_df = ibtracs_names_df.rename(
          columns={'ADT-HURSAT SID ': 'adt_hursat_sid',
                   'ALT SID        ':'alt_sid',
                   'ATCF ID   ':'atcf_id'})

      return ibtracs_names_df # Return the processed dataframe

  except Exception as e:
      print("ERROR READING IBTRACS SUPPORT FILE")
      traceback.print_exc()
      print(e)
      return None # Return None to indicate an error


def read_boundary_image(image_url):
  """
  Reads a boundary image from a Google Cloud Storage URL, converts it to a
  NumPy array, and returns the array and its dimensions. Boundary image is used
  to best identify locations for tropical cyclone basins in the North Atlantic Ocean
  and the Eastern North Pacific Ocean. Critical for use in the get_basin() function.

  Returns:
      numpy.ndarray: A NumPy array representing the image data. Returns None in
                     case of an error during fetching or processing.
  """
  try:
    # Fetch the image data from the URL
    response = requests.get(image_url)
    response.raise_for_status()  # Raise an exception for bad status codes

    # Create an in-memory file-like object from the response content
    image_bytes = io.BytesIO(response.content)

    # Open the image using PIL (Pillow library)
    pil_image = Image.open(image_bytes)

    # Convert the PIL Image to a NumPy array (height, width, channels)
    epac_natl_array = np.array(pil_image)

    # Print the shape of the image array and its dimensions
    print(f"Image array shape: {epac_natl_array.shape}")
    ny,nx = epac_natl_array.shape
    print(ny,nx)

    return epac_natl_array # Return the image as a NumPy array

  except requests.exceptions.RequestException as e:
      print(f"Error fetching image: {e}")
      traceback.print_exc()
      return None
  except IOError as e:
      print(f"Error processing image: {e}")
      traceback.print_exc()
      return None
  except Exception as e:
      print(f"An unexpected error occurred: {e}")
      traceback.print_exc()
      return None


def validate_and_normalize_month(month_input):
    """
    Validates a month input and returns it as an integer from 1 to 12.
    Accepts integers, numeric strings, month names, or abbreviations.
    """
    if isinstance(month_input, int):
        if 1 <= month_input <= 12:
            return month_input
        else:
            raise ValueError(f"Month as an integer must be between 1 and 12.")

    if not isinstance(month_input, str):
        raise ValueError(f"Input must be an integer or string, not {type(month_input).__name__}.")

    month_str = month_input.lower().strip()
    month_names = [name.lower() for name in calendar.month_name[1:]]
    month_abbrs = [abbr.lower() for abbr in calendar.month_abbr[1:]]

    try:
        if month_str in month_names:
            return month_names.index(month_str) + 1
        if month_str in month_abbrs:
            return month_abbrs.index(month_str) + 1
        month_int = int(month_str)
        if 1 <= month_int <= 12:
            return month_int
        else:
            raise ValueError
    except (ValueError, IndexError):
        raise ValueError(f"'{month_input}' is not a valid month.")


def validate_and_normalize_year(year_input):
    """
    Validates and normalizes a year input to a four-digit integer.

    Args:
        year_input: The year to validate (e.g., 2022 or "2022").

    Returns:
        The year as an integer.

    Raises:
        ValueError: If the input cannot be resolved to a valid year.
    """
    # Step 1: Try to convert the input to an integer.
    try:
        year = int(year_input)
    except (ValueError, TypeError):
        # This block now ONLY catches conversion errors.
        raise ValueError(f"'{year_input}' is not a valid year format.")

    # Step 2: Now that we have a number, check if it's in a valid range.
    current_year = datetime.now().year
    if not (1851 <= year <= current_year):
        # If this error is raised, it will be sent to the main function.
        raise ValueError(f"Year must be between 1851 and {current_year}.")

    return year

### --2.2-- ADT-HURSAT File and Storm Identification Functions

These functions are specific to accessing files in the Google Cloud Storage bucket, determining a storm's basin, and applying file-based filters.

In [ ]:
'''
2.2 ADT-HURSAT File and Storm Identification
'''

def get_list_adt_hursat_files():
  """
  Lists all file names within a specified Google Cloud Storage bucket and path,
  prefixes them with the storage URL, and returns the list of full file paths.

  Returns:
      list: A list of strings, where each string is the full Google Cloud Storage
            URL to an ADT-HURSAT file. Returns an empty list in case of an error.
  """
  try:
    # Create an anonymous client to access the public bucket
    storage_client = storage.Client.create_anonymous_client()

    # Define the bucket name and the path within the bucket
    adt_hursat_bucket_name = 'noaa-ncei-ipg'
    adt_hursat_path = 'datasets/hursat/adt/history-files/'

    # Get the bucket object
    bucket = storage_client.bucket(adt_hursat_bucket_name)

    # List all files within the specified path, using a delimiter for directory-like listing
    all_files_list = storage_client.list_blobs(bucket, prefix=adt_hursat_path, delimiter='/')

    # Construct the full Google Cloud Storage URL for each file
    full_file_path_list_tmp = []
    for file in all_files_list:
      full_file_path = f"https://storage.googleapis.com/{adt_hursat_bucket_name}/{file.name}"
      full_file_path_list_tmp.append(full_file_path)

    # Exclude the first element which is typically the directory itself due to delimiter
    full_file_path_list = full_file_path_list_tmp[1:]

    return full_file_path_list # Return the list of file paths

  except Exception as e:
    print('ERROR WITH GET LIST')
    traceback.print_exc()
    print(e)
    return [] # Return an empty list in case of an error



def get_basin(hemi, lat, lon, img_array):
    """
    Determines the tropical cyclone basin based on hemisphere, latitude, and longitude,
    using a boundary image to subdivide the NA and EP basins.

    Args:
        hemi (str): Hemisphere of the storm ('N' for Northern, 'S' for Southern).
        lat (float): Latitude of the storm in degrees north.
        lon (float): Longitude of the storm in degrees east.
        img_array(numpy array): Image array used to determine basins between
           North Atlantic or Eastern Pacific.

    Returns:
        str: The two-letter basin ID ('NA', 'EP', 'NI', 'SI', 'WP', 'SP', 'SA')
             or None if the basin cannot be determined.
    """
    try:
        # Initialize basin to None
        basin = None

        # Define boundaries and grid spacing for image array used to delineate between NA and EP based on IBTrACS logic
        dx = 0.1
        dy = 0.1
        x1 = -120 + 360 # Longitude range for a specific boundary check
        x2 = -75 + 360
        y1 = 0          # Latitude range for a specific boundary check
        y2 = 35
        ny = (y2 - y1) / dy + 1 # Number of grid points in y direction
        nx = (x2 - x1) / dx + 1 # Number of grid points in x direction

        if hemi == 'S':
            # Southern Hemisphere basin logic
            if (10 < lon < 135):
                basin = 'SI'  # South Indian
            elif (135 < lon < 290):
                basin = 'SP'  # Southern Pacific
            elif (290 < lon < 360) or (0 <= lon <= 10):
                basin = 'SA'  # South Atlantic
            else:
                basin = None # Undefined Basin in Southern Hemisphere

        elif hemi == 'N':
            # Northern Hemisphere basin logic
            if (30 < lon < 100):
                basin = 'NI'  # North Indian
            elif (100 <= lon < 180):
                basin = 'WP'  # Western North Pacific
            elif (180 <= lon <= 260):
                basin = 'EP'  # Eastern North Pacific
            elif (280 <= lon < 360) or (0 <= lon <= 30):
                basin = 'NA'  # North Atlantic
            elif (250 <= lon < 280):
                # Specific check for boundaries between NA and EP basins using the image
                if (lon >= x2) or (lon < 170):
                    basin = 'NA'
                elif (lon < y1):
                    basin = 'EP'
                else:
                    # Calculate image array indices corresponding to lat/lon
                    iy = round((lon - x1) / dx)
                    ix = round((lat - y1) / dy)

                    # Check bounds of indices before accessing the array
                    if (ix < 0) or (iy < 0) or (ix >= img_array.shape[0]) or (
                        iy >= img_array.shape[1]):
                        basin = None # Indices out of bounds, unable to determine basin
                        #print(f"Indices out of bounds: ix={ix}, iy={iy}")
                    else:
                        # Access pixel value from boundary map. The image must be flipped
                        # because PIL reads image in from top left (0,0), and IDL reads
                        # image in from the bottom left (0,0).
                        flipped_epac_natl_img_array = np.flipud(img_array)

                        # Ensure indices are within the flipped array bounds
                        flipped_ny, flipped_nx = flipped_epac_natl_img_array.shape
                        if ix < flipped_ny and iy < flipped_nx:
                            pixel_value = flipped_epac_natl_img_array[ix, iy]
                            #print(f"Value of the pixel at ({ix}, {iy}): {pixel_value}")
                            if (pixel_value > 200):
                                basin = 'EP'
                            elif (pixel_value > 100):
                                basin = 'NA'
                            else:
                                #print("pixel value is over land")
                                basin = None # Pixel value indicates land or unknown
                        else:
                            basin = None # Indices out of bounds for flipped array
                            #print(f"Flipped indices out of bounds: ix={ix}, iy={iy}")

            else:
                basin = None # Undefined Basin in Northern Hemisphere
        else:
            basin = None # Undefined Hemisphere

        return basin # Return the determined basin ID
    except Exception as e:
        print('ERROR WITH GET BASIN')
        traceback.print_exc()
        print(e)
        return None



def filter_data_list_by_basin(adt_hursat_files_path_list, basin_to_filter, epac_na_img_array):
  """
  Filters a list of ADT-HURSAT file paths to include only those files that
  correspond to a specific tropical cyclone basin, based on the storm's
  starting latitude and longitude extracted from the filename.

  Args:
    adt_hursat_files_path_list (list): A list of full file paths to ADT-HURSAT files.
    basin_to_filter (str): The two-letter basin ID to filter by (e.g., 'NA', 'EP').
    epac_na_img_array (numpy array): An array representing the byte-based image
        used to help id the North Atlantic and Eastern Pacific basins location.

  Returns:
    list: A list of file paths that are within the specified basin. Returns
          an empty list if an error occurs or no files match the filter.
  """
  basin_filtered_list = []
  try:
    # Iterate through each file path in the input list
    for file_path in adt_hursat_files_path_list:
        base_name = os.path.basename(file_path)
        file_name = os.path.splitext(os.path.basename(file_path))[0]

        # Process only files ending with '.nc'
        if base_name.endswith(".nc"):
            try:
                # Extract hemisphere, latitude, and longitude from the filename
                # Filename format is assumed to be YYYYDDDHemisphereLatLon.nc
                hemi = file_name[7]
                lat_int = file_name[8:10]
                lon_int = file_name[10:13]

                lat = int(lat_int)
                lon = int(lon_int)

                # Call the get_basin function to determine the basin
                basin = get_basin(hemi, lat, lon, epac_na_img_array)

                # If the determined basin matches the basin to filter, add the file path
                if basin is not None and basin == basin_to_filter:
                    basin_filtered_list.append(file_path)

            except (IndexError, ValueError) as e:
                # Handle cases where filename format is unexpected
                print(f"Skipping file due to unexpected filename format: {file_path} - {e}")
            except Exception as e:
                # Handle other potential errors during basin determination
                print(f"Error processing file {file_path}: {e}")
                traceback.print_exc()

    return basin_filtered_list # Return the list of filtered file paths

  except Exception as e:
    print('ERROR FILTERING BY BASIN')
    traceback.print_exc()
    print(e)
    return [] # Return an empty list in case of an error



def filter_by_month(files_list, target_month):
    """
    Filters a list of ADT-HURSAT file paths for a specific month.

    Args:
        files_list (list): A list of full file paths.
        target_month (any): The month to filter by (e.g., '01', 'January', 8).

    Returns:
        list: A list of file paths that fall within the target month.
    """
    mon_filtered_list = []
    try:
        # VALIDATE and NORMALIZE the user's input to a clean integer.
        month_to_find = validate_and_normalize_month(target_month)

        # Iterate through each file path in the input list
        for file_path in files_list:
            base_name = os.path.basename(file_path)
            file_name = os.path.splitext(base_name)[0]

            # Process only files ending with '.nc' and having a sufficient filename length
            if base_name.endswith(".nc") and len(file_name) >= 7:
                try:
                    # Extract year and day of the year from the filename
                    # Filename format is assumed to be YYYYDDDHLtLon.nc, where
                    # YYYY=Year, DDD=DayOfYear, H=Hemisphere, Lt= latitude, and Lon = longitude
                    year = int(file_name[0:4])
                    day_of_year = int(file_name[4:7])

                    first_day = datetime(year, 1, 1)
                    target_date = first_day + timedelta(days=day_of_year - 1)

                    # COMPARE the file's month (integer) directly to the validated month (integer).

                    if target_date.month == month_to_find:
                        mon_filtered_list.append(file_path)

                except (IndexError, ValueError) as e:
                    print(f"Skipping file due to date parsing error: {file_path} - {e}")
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")
                    traceback.print_exc()

    except ValueError as e:
        # This catches errors from the validation function if the initial input is bad.
        print(f"Error: Invalid month provided. {e}")
        return [] # Return an empty list on validation failure

    return mon_filtered_list



def filter_files_by_year(files_list, target_year):
    """
    Filters a list of file paths by year, validating the input year.

    Args:
        files_list (list): A list of file paths.
        target_year (any): The four-digit year to filter by (e.g., 2023 or '2023').

    Returns:
        list: A list of file paths filtered by the target year.
    """
    try:
        # VALIDATE and NORMALIZE the user's input to a clean integer.
        year_to_find = validate_and_normalize_year(target_year)
        year_to_find_str = str(year_to_find)

        # Note: If the user chooses a basin in the Southern Hemisphere (SH),
        # storms will still be split by calendar year, not the SH hurricane season
        # (July 1st to June 30th). The user would need to update this logic for seasonal analysis.

        filtered_list = []
        for file in files_list:
            filename = os.path.basename(file)

            # COMPARE the start of the filename with the validated string.
            # Filename format is assumed to be YYYYDDDHLtLon.nc, where
            # YYYY=Year, DDD=DayOfYear, H=Hemisphere, Lt= latitude, and Lon = longitude
            if filename.startswith(year_to_find_str):
                filtered_list.append(file)

        return filtered_list

    except ValueError as e:
        # This catches errors from the validation function if the initial input is bad.
        print(f"Error: Invalid year provided. {e}")
        return [] # Return an empty list on validation failure
    except Exception as e:
        # This is a general catch-all for other unexpected errors.
        print(f"An unexpected error occurred: {e}")
        traceback.print_exc()
        return []



def get_basin_name(basin_id):
  """
  Returns the full name of a tropical cyclone basin based on its ID.

  Args:
    basin_id (str): The two-letter ID of the basin (e.g., 'NA').

  Returns:
    str: The full name of the basin, or a message indicating the ID was
         not found.
  """
  try:
    # Dictionary mapping basin IDs to their full names
    basins = {
        'NA': 'North Atlantic',
        'SA': 'South Atlantic',
        'NI': 'North Indian',
        'SI': 'South Indian',
        'WP': 'Western North Pacific',
        'EP': 'Eastern North Pacific',
        'SP': 'Southern Pacific'
                }
    # Use the get() method to return the full name if the ID is found,
    # otherwise return the default message.
    return basins.get(basin_id, "Basin ID not found.")

  except Exception as e:
    print('ERROR GETTING BASIN NAME')
    traceback.print_exc()
    print(e)



def filter_ibtracs_dataframe_by_basin(ibtracs_dataframe, basin_to_filter):
    """
    Filters a single IBTrACS dataframe by a specified basin.

    Args:
        ibtracs_dataframe (pandas.DataFrame): A DataFrame containing IBTrACS data.
        basin_to_filter (str): The two-letter basin code to filter by (e.g., 'NA', 'EP').

    Returns:
        pandas.DataFrame: A DataFrame filtered by the specified basin. Returns
                          an empty DataFrame if the 'BASIN' column is missing
                          or if an error occurs.
    """
    try:
        # Ensure the 'BASIN' column exists before attempting to filter
        if 'BASIN' in ibtracs_dataframe.columns:
             # Filter for the specified basin and handle potential NaNs in the 'BASIN' column
             # Using .copy() to avoid SettingWithCopyWarning
             return ibtracs_dataframe[ibtracs_dataframe['BASIN'].dropna() == basin_to_filter].copy()
        else:
             print("BASIN column not found in the IBTRACS dataframe.")
             return pd.DataFrame() # Return an empty dataframe if 'BASIN' column is missing

    except Exception as e:
        print('Error filtering IBTRACS dataframe by basin')
        traceback.print_exc()
        print(e)
        return pd.DataFrame() # Return an empty dataframe in case of an error



def process_adt_file(file_path):
    """
    Helper function to process a single ADT-HURSAT NetCDF file from a Google Cloud
    Storage URL, load it into an xarray Dataset, convert it to a pandas DataFrame,
    and add a 'storm_id' column extracted from the filename.

    Args:
        file_path (str): The full Google Cloud Storage URL to the ADT-HURSAT file.

    Returns:
        pandas.DataFrame: A DataFrame containing the data from the NetCDF file
                          with an added 'storm_id' column. Returns None if an
                          error occurs during processing.
    """
    try:
        # Use fsspec to open the file from the Google Cloud Storage URL
        with fsspec.open(file_path, 'rb') as f:
            # Open the dataset using xarray
            ds = xr.open_dataset(f)

            # Convert the xarray Dataset to a pandas DataFrame
            df = ds.to_dataframe()

            # Extract the storm_id from the filename
            base_name = os.path.basename(file_path)
            storm_id = os.path.splitext(os.path.basename(base_name))[0]

            # Add the storm_id as a new column to the DataFrame
            df['storm_id'] = storm_id

            return df # Return the processed DataFrame
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        # Return None to indicate that processing for this file failed
        return None



def build_adt_hursat_dataframe_parallel(basin_list_to_ingest):
    """
    Builds a single pandas DataFrame by concurrently processing multiple ADT-HURSAT
    NetCDF files from a list of file paths. It utilizes a ThreadPoolExecutor
    for parallel processing and concatenates the resulting DataFrames.

    Args:
        basin_list_to_ingest (list): A list of full file paths to ADT-HURSAT files
                                     to be ingested into the DataFrame.

    Returns:
        pandas.DataFrame: A concatenated DataFrame containing data from all
                          successfully processed ADT-HURSAT files. Returns
                          None if an error occurs during the process.
    """
    try:
        adt_df_list = [] # List to store DataFrames from each processed file

        # Use ThreadPoolExecutor for parallel processing. Adjust max_workers
        # based on system capabilities and task nature.
        with ThreadPoolExecutor(max_workers=10) as executor:
            # Submit the process_adt_file function for each file path
            future_to_file = {executor.submit(
                process_adt_file, file_path): file_path for file_path in basin_list_to_ingest}

            # Process results as they complete
            for future in as_completed(future_to_file):
                file_path = future_to_file[future]
                try:
                    # Get the result of the submitted task (the DataFrame or None)
                    df = future.result()
                    if df is not None:
                        adt_df_list.append(df) # Add successful DataFrames to the list
                except Exception as exc:
                    # Print an error message if processing a specific file failed
                    print(f'{file_path} generated an exception: {exc}')

        # Concatenate all DataFrames in the list into a single DataFrame
        adt_hursat_compiled_df = pd.concat(adt_df_list, ignore_index=True)

        # Select and copy a subset of columns for a smaller DataFrame
        # User Note: You can choose to keep other column names depending on your desired use case for the data.
        adt_hursat_cols_keep = ['Date', 'Time','WindSpeed','Lat','Lon',
                                'MSLP','storm_id','EyeSize','RMW']
        adt_hursat_basin_small_all_hours_df = adt_hursat_compiled_df[adt_hursat_cols_keep].copy()

        return adt_hursat_basin_small_all_hours_df # Return the final concatenated DataFrame

    except Exception as e:
        print("ERROR BUILDING THE DATAFRAME")
        traceback.print_exc()
        print(e)
        return None # Return None to indicate an error

### --2.3-- Core Data Transformation and Merging Functions

This section handles the main data preparation steps, critical for aligning the disparate datasets:
 - **Temporal Standardization:** Rounding observations to the nearest 3-hour mark (3H) to minimize time shift errors.
 - **Filtering:** Selecting only synoptic times (00, 06, 12, 18Z).
 - **Merging:** Joining ADT-HURSAT and IBTrACS datasets based on standardized Storm ID (SID) and Time.
 - **Threshold Calculations:** Computing intensity metrics for analysis.

In [ ]:
'''
2.3 Core Data Transformation and Merging Functions
'''


def transform_adt_hursat_dataframe(adt_hursat_basin_df, ibtracs_names_df,
                                   basin_to_match):
  """
  Transforms the ADT-HURSAT basin DataFrame by creating a combined datetime column,
  rounding datetime to the nearest hour, filtering for specific hours, extracting
  the year, calculating storm counts exceeding intensity thresholds by year,
  and merging these counts into a summary DataFrame.

  Args:
    adt_hursat_basin_df (pandas.DataFrame): DataFrame containing ADT-HURSAT data
                                            for a specific basin.
    ibtracs_names_df (pandas.DataFrame): DataFrame containing IBTrACS names and
                                          supporting information (currently not
                                          directly used in transformations but
                                          kept as per original function signature).
    basin_to_match (str): The two-letter basin code (currently not directly
                          used in transformations but kept as per original
                          function signature).

  Returns:
    tuple: A tuple containing:
           - pandas.DataFrame: A summary DataFrame with counts of named storms,
                               hurricanes, and major hurricanes by year.
           - pandas.DataFrame: The transformed ADT-HURSAT DataFrame filtered
                               by specific hours.
           Returns None for both DataFrames if an error occurs.
  """
  try:
    # Select and copy a subset of columns
    adt_hursat_basin_small_all_hours_df = adt_hursat_basin_df[['Date', 'Time',
                                                              'WindSpeed','Lat',
                                                              'Lon','MSLP',
                                                              'storm_id',
                                                              'EyeSize','RMW']].copy()

    # Concatenate 'Date' and 'Time' columns to create a datetime string
    adt_hursat_basin_small_all_hours_df['datetime_str'] = (
        adt_hursat_basin_small_all_hours_df['Date'].astype(str) + ' ' +
        adt_hursat_basin_small_all_hours_df['Time'].astype(str))

    # Convert the combined string to datetime objects, coercing errors
    adt_hursat_basin_small_all_hours_df['datetime'] = pd.to_datetime(
        adt_hursat_basin_small_all_hours_df['datetime_str'],
        format='%Y%b%d %H%M%S', errors='coerce')

    # Round the datetime to the nearest 3 hour mark
    adt_hursat_basin_small_all_hours_df['rounded_datetime'] = (
        adt_hursat_basin_small_all_hours_df['datetime'].dt.round('3h'))

    # Filter out rows not corresponding to the desired hours (00, 06, 12, 18)
    desired_hours = [0, 6, 12, 18]
    adt_hursat_basin_small_df = adt_hursat_basin_small_all_hours_df[
        adt_hursat_basin_small_all_hours_df['rounded_datetime'].dt.hour.isin(
            desired_hours)].copy()

    # Set 'rounded_datetime' as the index
    adt_hursat_basin_small_df.set_index('rounded_datetime', inplace=True)

    # Extract year as integer, potential NaT will result in NaN which is handled by Int64Dtype
    adt_hursat_basin_small_df.loc[:, 'Year'] = adt_hursat_basin_small_df.index.year
    adt_hursat_basin_small_df['Year'] = adt_hursat_basin_small_df['Year'].astype(pd.Int64Dtype())

    # Define intensity thresholds in knots
    major_threshold = 96
    hurr_threshold = 64
    named_threshold = 34

    # Strip leading/trailing whitespace from 'storm_id'
    adt_hursat_basin_small_df.loc[:, 'storm_id'] = (
        adt_hursat_basin_small_df['storm_id'].str.strip())

    # Calculate if WindSpeed meets or exceeds major hurricane threshold
    adt_hursat_basin_small_df['over_threshold_major'] = (
        adt_hursat_basin_small_df['WindSpeed'].ge(major_threshold))
    # Group by storm_id and Year, and check if any observation for a storm in a year is over the major threshold
    major_grouped_df = adt_hursat_basin_small_df.groupby(
        ['storm_id', 'Year'])['over_threshold_major'].any().reset_index()
    # Count the number of storms per year that had at least one observation over the major threshold
    count_over_major_threshold_by_year = major_grouped_df[
        major_grouped_df['over_threshold_major']].groupby('Year').size(
        ).reset_index(name='Count_Over_Threshold_major')

    # Calculate if WindSpeed meets or exceeds hurricane threshold
    adt_hursat_basin_small_df['over_threshold_hurr'] = (
        adt_hursat_basin_small_df['WindSpeed'].ge(hurr_threshold))
    # Group by storm_id and Year, and check if any observation for a storm in a year is over the hurricane threshold
    hurr_grouped_df = adt_hursat_basin_small_df.groupby(
        ['storm_id', 'Year'])['over_threshold_hurr'].any().reset_index()
    # Count the number of storms per year that had at least one observation over the hurricane threshold
    count_over_hurr_threshold_by_year = hurr_grouped_df[
        hurr_grouped_df['over_threshold_hurr']].groupby('Year').size(
        ).reset_index(name='Count_Over_Threshold_hurr')

    # Calculate if WindSpeed meets or exceeds named storm threshold
    adt_hursat_basin_small_df['over_threshold_named'] = (
        adt_hursat_basin_small_df['WindSpeed'].ge(named_threshold))
    # Group by storm_id and Year, and check if any observation for a storm in a year is over the named threshold
    named_grouped_df = adt_hursat_basin_small_df.groupby(
        ['storm_id', 'Year'])['over_threshold_named'].any().reset_index()
    # Count the number of storms per year that had at least one observation over the named threshold
    count_all_named_by_year = named_grouped_df[
        named_grouped_df['over_threshold_named']].groupby('Year').size(
        ).reset_index(name='Count_Over_Threshold_named')

    # Merge the counts for named storms, hurricanes, and major hurricanes by year
    merged_df = pd.merge(count_all_named_by_year,
                         count_over_hurr_threshold_by_year, on='Year', how='left')
    merged_df = pd.merge(merged_df, count_over_major_threshold_by_year, on='Year',
                         how='left')

    # Rename columns for clarity
    merged_df.rename(columns={'Year':'Year',
                              'Count_Over_Threshold_named': 'Named_Storms',
                              'Count_Over_Threshold_hurr': 'Hurricanes',
                              'Count_Over_Threshold_major': 'Major_Hurricanes'},
                     inplace=True)

    # Set 'Year' as the index
    merged_df = merged_df.set_index('Year')

    return merged_df, adt_hursat_basin_small_df # Return both dataframes

  except Exception as e:
    print('ERROR TRANSFORMING DATAFRAME')
    traceback.print_exc()
    print(e)
    return None, None # Return None to indicate an error


def build_adt_hursat_ibtracs_graphs_dataframe(adt_hursat_basin_small_df,
                                              ibtracs_1978_2024_small_df):
  """
  Merges transformed ADT-HURSAT and IBTrACS DataFrames on common columns
  ('SID', 'date', 'hour') to create a combined DataFrame suitable for comparison
  plots. It also renames columns for consistency and performs datetime handling.

  Args:
    adt_hursat_basin_small_df (pandas.DataFrame): Transformed ADT-HURSAT DataFrame
                                                 (filtered by hour, with datetime index).
    ibtracs_1978_2024_small_df (pandas.DataFrame): Filtered IBTrACS DataFrame
                                                 (years 1978-2024).

  Returns:
    pandas.DataFrame: A merged DataFrame containing data from both ADT-HURSAT and
                      IBTrACS, indexed by datetime, and filtered for specific hours.
                      Returns None if an error occurs.
  """
  try:
    # Reset the index of the ADT-HURSAT DataFrame to make the datetime column
    # available for merging
    adt_hursat_basin_small_df.reset_index(inplace=True)

    # Rename columns in the IBTrACS DataFrame to match ADT-HURSAT for merging
    ibtracs_1978_2024_small_df.rename(
        columns={
            'LAT_degrees_north': 'Lat',
            'LON_degrees_east': 'Lon',
            'USA_WIND_kts': 'WindSpeed',
            'USA_PRES_mb': 'MSLP',
            'USA_RMW_nmile': 'RMW',
            'USA_EYE_nmile': 'EyeSize',
        },
        inplace=True,
    )

    # Rename the 'storm_id' column in the ADT-HURSAT DataFrame to 'SID' for merging
    adt_hursat_basin_small_df.rename(columns={'storm_id': 'SID'}, inplace=True)

    # Problems converting the ISO_TIME resulted in this set of modifications
    # Remove milliseconds to match the required formatting later
    ibtracs_1978_2024_small_df['ISO_TIME_truncated'] = (
        ibtracs_1978_2024_small_df['ISO_TIME'].astype(str).str.split('.').str[0]
    )
    # Now convert the truncated string column to datetime using the original format
    ibtracs_1978_2024_small_df['ISO_TIME'] = pd.to_datetime(
        ibtracs_1978_2024_small_df['ISO_TIME_truncated'],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce',
    )
    # Drop the temporary truncated column
    ibtracs_1978_2024_small_df.drop(columns=['ISO_TIME_truncated'], inplace=True)

    # Convert 'WindSpeed' column in IBTrACS to numeric, coercing errors to NaN
    ibtracs_1978_2024_small_df['WindSpeed'] = pd.to_numeric(
        ibtracs_1978_2024_small_df['WindSpeed'], errors='coerce'
    )

    # Extract date and hour components from datetime columns for merging
    ibtracs_1978_2024_small_df['date'] = ibtracs_1978_2024_small_df['ISO_TIME'].dt.date
    ibtracs_1978_2024_small_df['hour'] = ibtracs_1978_2024_small_df['ISO_TIME'].dt.hour

    adt_hursat_basin_small_df['date'] = (
        adt_hursat_basin_small_df['rounded_datetime'].dt.date
    )
    adt_hursat_basin_small_df['hour'] = (
        adt_hursat_basin_small_df['rounded_datetime'].dt.hour
    )

    # Define the common columns for merging
    common_columns = ['SID', 'date', 'hour']

    # Merge the two dataframes based on common columns, keeping all records that match
    merged_allrecs_ibtracs_adthursat_df = pd.merge(
        ibtracs_1978_2024_small_df,
        adt_hursat_basin_small_df,
        on=common_columns,
        how='inner',
        suffixes=('_ibtracs', '_adt_hursat'),
    )

    # Convert 'ISO_TIME' column to datetime objects in the merged dataframe
    merged_allrecs_ibtracs_adthursat_df['datetime'] = pd.to_datetime(
        merged_allrecs_ibtracs_adthursat_df['ISO_TIME'],
        format='%Y%b%d %H%M%S',
        errors='coerce',
    )

    # Filter out rows not corresponding to the desired hours (00, 06, 12, 18) in the merged dataframe
    desired_hours = [0, 6, 12, 18]
    merged_ibtracs_adthursat_df = merged_allrecs_ibtracs_adthursat_df[
        merged_allrecs_ibtracs_adthursat_df['datetime'].dt.hour.isin(desired_hours)
    ]

    # Set the 'datetime' column as the index for the final merged dataframe
    merged_ibtracs_adthursat_df.set_index('datetime', inplace=True)

    return merged_ibtracs_adthursat_df  # Return the final merged and filtered dataframe

  except Exception as e:
    print("ERROR IN BUILDING DATAFRAME")
    traceback.print_exc()
    print(e)
    return None # Return None to indicate an error


### --2.4-- Accessibility Helpers (Plotting Support)
These utility functions ensure the visualization outputs meet accessibility standards by generating descriptive, detailed Alt Text. They handle tasks like text wrapping and formatting acronyms for screen readers (e.g., handling storm metadata based on when a storm started).

In [ ]:
'''
2.4 Accessibility Helpers (Plotting Support)
'''


def simple_word_wrap(text, wrap_width=80):
    """Wraps a given text at word breaks to a specified width."""
    try:
        if not isinstance(text, str):
            return text
        wrapper = textwrap.TextWrapper(width=wrap_width, break_on_hyphens=False)
        wrapped_text = '\n'.join(wrapper.wrap(text))
        return wrapped_text
    except Exception as e:
        print('ERROR FILTERING BY YEAR')
        traceback.print_exc()
        print(e)


def simple_text_wrap(text, wrap_width=80):
    """Wraps a given text at character breaks to a specified width."""
    try:

        if not isinstance(text, str):
            return text
        wrapper = textwrap.TextWrapper(
            width=wrap_width, break_on_hyphens=False, break_long_words=True)
        wrapped_text = '\n'.join(wrapper.wrap(text))
        return wrapped_text
    except Exception as e:
        print('ERROR FILTERING BY YEAR')
        traceback.print_exc()
        print(e)


def format_acronyms_for_screen_reader(text_to_format):
    """
    Formats specific acronyms in a string to be read letter by letter
    by a screen reader by inserting spaces between characters.

    Args:
        text_to_format (str): The string to format.

    Returns:
        str: The formatted string.
    """
    acronyms = {
        'ADT-HURSAT': 'A D T H U R S A T',
        'IBTrACS': 'I B T r A C S',
    }

    formatted_text = text_to_format
    for acronym, spaced_out in acronyms.items():
        formatted_text = formatted_text.replace(acronym, spaced_out)

    return formatted_text

### --2.5-- Visualization Generation Functions
This section contains all the plotting functions, each generating a specific chart and constructing the necessary detailed Alt Text

In [ ]:
'''
2.5 Visualization Generation Functions
'''

def generate_bar_chart_adt_hursat_storm_counts(
    adt_hursat_counts_df, basin, params):
    """
    Generates a grouped bar chart of ADT-HURSAT storm counts by year
    with alt text.

    Args:
        adt_hursat_counts_df: DataFrame containing storm counts by year.
            Expected columns: 'Named_Storms', 'Hurricanes', 'Major_Hurricanes'.
            Index should be the year.
        basin: The basin name for the plot title.
        params: Dictionary of plotting parameters.
            Includes: figsize, color_map, bar_width, group_spacing, title,
                      xlabel, ylabel, xticks_rotation, xticks_ha,
                      legend_title.

    Returns:
        A dictionary containing the figure object, filename, and alt text string.
        Returns None if data is empty or missing required columns.
    """
    alt_text_string = "Alternative text could not be generated."

    try:
        adt_hursat_bar_chart_df = adt_hursat_counts_df.copy()

        if adt_hursat_bar_chart_df.empty:
            print(f"No data available to generate the bar chart for the {basin} Basin.")
            alt_text_string = (
                f"Bar chart of storm counts for the {basin} Basin: No data"
                " available for the selected parameters.")
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {'filename': 'adt_hursat_storm_counts_barchart.png',
                    'figure': None, 'alt_text': alt_text_string}

        required_cols = ['Named_Storms', 'Hurricanes', 'Major_Hurricanes']
        if not all(col in adt_hursat_bar_chart_df.columns for col in required_cols):
            print(
                "Error: Input dataframe for Bar Chart is missing required"
                f" columns: {', '.join(required_cols)}.")
            alt_text_string = (
                f"Bar chart of storm counts for the {basin} Basin: Missing"
                " required data columns.")
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {'filename': 'adt_hursat_storm_counts_barchart.png',
                    'figure': None, 'alt_text': alt_text_string}

        fig, ax = plt.subplots(figsize=params.get('figsize', (12, 6)))

        bar_width = params.get('bar_width', 0.2)
        group_spacing = params.get('group_spacing', 0.05)
        num_categories = len(required_cols)
        total_group_width = bar_width * num_categories
        # Total width of all bars in a group plus spacing between them
        total_width_with_spacing = total_group_width + (
            num_categories - 1) * group_spacing
        # Offset to center the group on the x-tick
        group_center_offset = total_group_width / 2 # Adjusted for center

        colors = params.get(
            'color_map', {
                'Named_Storms': '#1b9e77',
                'Hurricanes': '#d95f02',
                'Major_Hurricanes': '#7570b3'
            })

        years = adt_hursat_bar_chart_df.index
        x = np.arange(len(years))

        # Calculate x positions for each category's bars
        x_pos_named_storms = x - group_center_offset + bar_width / 2
        x_pos_hurricanes = x - group_center_offset + bar_width + group_spacing + bar_width / 2
        x_pos_major_hurricanes = x - group_center_offset + 2 * bar_width + 2 * group_spacing + bar_width / 2

        # Plotting the bars side-by-side
        ax.bar(
            x_pos_named_storms,
            adt_hursat_bar_chart_df['Named_Storms'],
            bar_width,
            label='Named Storms',
            color=colors.get('Named_Storms', '#1b9e77'))
        ax.bar(
            x_pos_hurricanes,
            adt_hursat_bar_chart_df['Hurricanes'],
            bar_width,
            label='Hurricanes',
            color=colors.get('Hurricanes', '#d95f02'))
        ax.bar(
            x_pos_major_hurricanes,
            adt_hursat_bar_chart_df['Major_Hurricanes'],
            bar_width,
            label='Major Hurricanes',
            color=colors.get('Major_Hurricanes', '#7570b3'))

      # Add horizontal lines based on parameters
        for line_params in params.get('horizontal_lines', []):
            ax.axhline(**line_params)

        ax.set_title(params.get('title', f'ADT-HURSAT Storm Counts, {basin} Basin'))
        ax.set_xlabel(params.get('xlabel', 'Year'))
        ax.set_ylabel(params.get('ylabel', 'Annual Count'))
        ax.legend(title=params.get('legend_title', 'Category'))

        # Set x-axis ticks and labels to show every year
        ax.set_xticks(x)
        ax.set_xticklabels(years)
        plt.xticks(rotation=params.get('xticks_rotation', 45),
               ha=params.get('xticks_ha', 'right'),
               rotation_mode='anchor')

        # Set major and minor tick spacing on the y-axis
        major_tick_spacing = params.get('major_tick_spacing', None)
        minor_tick_spacing = params.get('minor_tick_spacing', None)

        if major_tick_spacing is not None:
            ax.yaxis.set_major_locator(ticker.MultipleLocator(major_tick_spacing))
        if minor_tick_spacing is not None:
            ax.yaxis.set_minor_locator(ticker.MultipleLocator(minor_tick_spacing))

        # Ensure y-axis ticks are integers
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%d'))
        ax.yaxis.set_minor_formatter(ticker.FormatStrFormatter('%d'))

        # Set y-axis tick parameters for major ticks (rotation 90)
        ax.tick_params(axis='y', which='major', rotation=90)
        # Set y-axis tick parameters for minor ticks (rotation 90, smaller font size)
        ax.tick_params(axis='y', which='minor', rotation=90, labelsize=params.get('minor_yticks_fontsize', 8))

        ax.tick_params(which='minor', axis='y', length=4, width=1, direction='out')

        # Set x-axis limits to match the data range precisely
        first_group_left_edge = x[0] - group_center_offset - group_spacing / 2
        last_group_right_edge = x[-1] + group_center_offset + group_spacing / 2
        ax.set_xlim(first_group_left_edge, last_group_right_edge)

        # Add grid lines for both major and minor ticks
        ax.grid(which='major', axis='y', linestyle='-', linewidth='0.5', color='gray')
        ax.grid(which='minor', axis='y', linestyle=':', linewidth='0.5', color='gray')

        plt.tight_layout()

        # --- Construct Alt Text ---
        try:
            #use helper function to get the acronym properly formatted for alt text
            original_title = ax.get_title()
            alt_text_title = format_acronyms_for_screen_reader(original_title)

            alt_text_parts = [
                f"Grouped bar chart titled '{alt_text_title or 'Storm Counts Bar Chart'}'.",
                f"The x-axis represents Year from {years.min()} to {years.max()}.",
                f"The y-axis represents Annual Count.",
                (
                    "The bars show the annual counts of named storms, hurricanes,"
                    " and major hurricanes."
                ),
            ]

            for column in required_cols:
                if (column in adt_hursat_bar_chart_df.columns and
                        not adt_hursat_bar_chart_df[column].dropna().empty):
                    category_name = column.replace('_', ' ')
                    category_color = colors.get(column, '#666666')
                    color_map = {
                        '#1b9e77': 'green',
                        '#d95f02': 'orange',
                        '#7570b3': 'dark blue',
                        '#666666': 'dark gray',
                    }
                    category_color_name = color_map.get(
                        category_color, category_color)

                    min_count = adt_hursat_bar_chart_df[column].min()
                    max_count = adt_hursat_bar_chart_df[column].max()
                    avg_count = adt_hursat_bar_chart_df[column].mean()
                    total_count = adt_hursat_bar_chart_df[column].sum()

                    alt_text_parts.append(
                        f"{category_name} are represented in {category_color_name}. Annual"
                        f" counts range from {min_count:.0f} to {max_count:.0f}, with an"
                        f" average of {avg_count:.1f}. Total count for this category is"
                        f" {total_count:.0f}.")
                else:
                    category_name = column.replace('_', ' ')
                    alt_text_parts.append(f"{category_name}: No data available.")

            alt_text_string = " ".join(alt_text_parts)
            formatted_alt_text = simple_word_wrap(alt_text_string, wrap_width=80)
            print(f"Alt text: {formatted_alt_text}")

        except Exception as manual_alt_text_e:
            print(f"Error generating alt text: {manual_alt_text_e}")
            traceback.print_exc()
            alt_text_string = (
                "Alternative text could not be generated due to an error.")
            print(f"Alt text: {alt_text_string}")

        fig_caption = (
            'Figure Caption: This plot shows storm counts per year, separated into groups of named storms '
            '(>34 kts), hurricanes (>64 kts) and major hurricanes (>96 kts) for the ADT-HURSAT data. Note that '
            'ADT-HURSAT storm counts are dependent on IBTrACS, which was used to initially identify '
            'storms in HURSAT, however there may be storms from IBTrACS, particularly in the earlier '
            'years, that do not show up in ADT-HURSAT due to a lack of availability in satellite data at the '
            'time and location. Fluctuations in North Atlantic hurricane counts from decade to decade have '
            'been well documented in literature as Atlantic Multidecadal Variability, though attribution of the '
            'phenomena (likely a combination of internal variability and aerosols) is still not widely agreed '
            'upon. Counts of named storms are a less reliable metric due to changes over time and between '
            'agencies in reporting practices for weak and short-lived storms.'
            )

        plt.show()

        return {
            'filename': 'adt_hursat_storm_counts_barchart.png',
            'figure': fig,
            'alt_text': alt_text_string,
            'caption' : fig_caption
        }

    except Exception as e:
        print('ERROR WITH BAR CHART')
        traceback.print_exc()
        alt_text_string = "Alternative text could not be generated due to an error."
        print(f"Alt text: {simple_word_wrap(alt_text_string)}")
        return {'filename': 'adt_hursat_storm_counts_barchart.png',
                'figure': None, 'alt_text': alt_text_string}



def generate_proportional_intensities_adt_hursat_time_series(
    adt_hursat_dataframe, basin, params):
    """
    Generates a time series plot of proportional major hurricane fixes
    from ADT-HURSAT data with alt text.

    LOGIC CLARIFICATION:
    This function performs BINNING, not averaging.
    1. Resamples data into time bins (e.g., '1YE' for 1 Year).
    2. Counts total fixes in each bin.
    3. Counts fixes >= Major Threshold in each bin.
    4. Counts fixes >= Hurricane Threshold in each bin.
    5. Calculates Proportion = (Major Count) / (Hurricane Count).

    Args:
      adt_hursat_dataframe: DataFrame containing ADT-HURSAT data.
          Expected columns: 'rounded_datetime', 'WindSpeed', 'SID'.
      basin: The basin name for the plot title.
      params: Dictionary of plotting parameters.
          Includes: figsize, resample_period, major_threshold, hurr_threshold,
                    line_color, marker_color, trend_line_color,
                    trend_line_linestyle, xlabel, ylabel, title, legend_loc,
                    legend_bbox_to_anchor, legend_ncol, legend_fontsize, grid,
                    xticks_rotation, xticks_ha, xticks_fontsize.

    Returns:
      A dictionary containing the figure object, filename, and alt text string.
      Returns None if data is empty or missing required columns.
    """
    alt_text_string = "Alternative text could not be generated."

    try:
        adt_hursat_prop_int_dataframe = adt_hursat_dataframe.copy()

        # Reset index to make 'rounded_datetime' a column for resampling if it's
        # the index
        if isinstance(adt_hursat_prop_int_dataframe.index, pd.DatetimeIndex):
             adt_hursat_prop_int_dataframe.reset_index(inplace=True)

        if adt_hursat_prop_int_dataframe.empty:
            print(
                "No data available to generate the proportional intensities time"
                f" series for the {basin} Basin.")
            alt_text_string = (
                "Proportional intensities time series plot for the"
                f" {basin} Basin: No data available for the selected parameters.")
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {
                'filename': 'proportional_intensities_time_series.png',
                'figure': None,
                'alt_text': alt_text_string
            }

        major_threshold = params.get('major_threshold', 96)
        hurr_threshold = params.get('hurr_threshold', 64)
        resample_period = params.get('resample_period', '1YE')

        if 'rounded_datetime' not in adt_hursat_prop_int_dataframe.columns:
            print("Error: 'rounded_datetime' column not found in the dataframe.")
            alt_text_string = (
                "Proportional intensities time series plot for the"
                f" {basin} Basin: Missing required data columns.")
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {
                'filename': 'proportional_intensities_time_series.png',
                'figure': None,
                'alt_text': alt_text_string
            }

        adt_hursat_prop_int_dataframe['rounded_datetime'] = pd.to_datetime(
            adt_hursat_prop_int_dataframe['rounded_datetime'], errors='coerce')

        # Create a copy with 'rounded_datetime' as index for resampling, then
        # reset index
        # NOTE: This uses sum() and size, confirming it relies on BINNING (counts),
        # not averaging.
        resampled_data_df = adt_hursat_prop_int_dataframe.set_index(
            'rounded_datetime').resample(resample_period)['WindSpeed'].agg(
                total_count='size',
                major_fixes=lambda x: (x >= major_threshold).sum(),
                total_hurricane_fixes=lambda x: (x >= hurr_threshold).sum()
            ).reset_index()

        if resampled_data_df.empty:
            print(
                "No resampled data available to generate the time series for the"
                f" {basin} Basin.")
            alt_text_string = (
                "Proportional intensities time series plot for the"
                f" {basin} Basin: No resampled data available for the selected"
                " parameters.")
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {
                'filename': 'proportional_intensities_time_series.png',
                'figure': None,
                'alt_text': alt_text_string
            }

        # Handle division by zero safely
        resampled_data_df['proportion_over_or_equal_threshold'] = resampled_data_df.apply(
            lambda row: row['major_fixes'] / row['total_hurricane_fixes']
            if row['total_hurricane_fixes'] > 0 else np.nan, axis=1
        )

        trend_data_df = resampled_data_df.dropna(
            subset=['proportion_over_or_equal_threshold']).copy()

        if 'Year' in trend_data_df.columns and not trend_data_df['Year'].isnull().all():
            years_for_linregress = trend_data_df['Year'].dropna().astype(int)
        else:
            years_for_linregress = (
                trend_data_df['rounded_datetime'].dt.year.dropna().astype(int))

        trend_line = []
        slope = None
        if len(years_for_linregress) > 1:
            try:
                slope, intercept, r_value, p_value, std_err = linregress(
                    years_for_linregress,
                    trend_data_df.loc[years_for_linregress.index,
                                      'proportion_over_or_equal_threshold'].values)
                trend_line = slope * (
                    resampled_data_df['rounded_datetime'].dt.year) + intercept
            except Exception as linregress_e:
                print(f"Error calculating trend line: {linregress_e}")

        #Attempt to get separate dataframe for plotting and alt text
        axis_df = resampled_data_df.copy()
        axis_df['Years'] = axis_df['rounded_datetime'].dt.year

        #Build the plot
        fig, ax = plt.subplots(figsize=params.get('figsize', (14, 6)))

        ax.plot(
            axis_df['Years'],
            resampled_data_df['proportion_over_or_equal_threshold'],
            label=f'{resample_period} Bins', # "Bins" is correct for counting logic
            color=params.get('line_color', '#0076D6'))
        ax.plot(
            axis_df['Years'],
            resampled_data_df['proportion_over_or_equal_threshold'],
            'o',
            color=params.get('marker_color', '#0076D6'))

        ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

        if len(trend_line) > 0:
            ax.plot(
                resampled_data_df['rounded_datetime'].dt.year,
                trend_line,
                linestyle=params.get('trend_line_linestyle', '--'),
                color=params.get('trend_line_color', '#707070'),
                label='Trend Line')

        ax.set_xlabel(params.get('xlabel', 'Year'))
        ax.set_ylabel(params.get('ylabel', 'Proportion of major hurricane fixes to all hurricane fixes'))
        ax.set_title(
            params.get('title',
                       f'Proportion of major hurricane fixes to all hurricane fixes,'
                       f' {basin} Basin'))
        ax.legend(
            loc=params.get('legend_loc', 'lower center'),
            bbox_to_anchor=params.get('legend_bbox_to_anchor', (0.5, -0.3)),
            ncol=params.get('legend_ncol', 6),
            fontsize=params.get('legend_fontsize', 7))
        ax.grid(params.get('grid', True))
        plt.xticks(
            rotation=params.get('xticks_rotation', 45),
            ha=params.get('xticks_ha', 'right'),
            fontsize=params.get('xticks_fontsize', 7))
        plt.tight_layout()


        # --- Construct Alt Text ---
        try:
            #use helper function to get the acronym properly formatted for alt text
            original_title = ax.get_title()
            alt_text_title = format_acronyms_for_screen_reader(original_title)

            min_prop_data = resampled_data_df['proportion_over_or_equal_threshold'].min()
            max_prop_data = resampled_data_df['proportion_over_or_equal_threshold'].max()
            avg_prop_data = resampled_data_df['proportion_over_or_equal_threshold'].mean()
            min_prop_year_data = resampled_data_df.loc[
                resampled_data_df['proportion_over_or_equal_threshold'].idxmin(),
                'rounded_datetime'].year
            max_prop_year_data = resampled_data_df.loc[
                resampled_data_df['proportion_over_or_equal_threshold'].idxmax(),
                'rounded_datetime'].year

            alt_text_parts = [
                f"Time series plot titled '{alt_text_title or 'Proportional Intensities Time Series'}'.",
                (
                    f"The x-axis represents Year from"
                    f" {resampled_data_df['rounded_datetime'].dt.year.min()} to"
                    f" {resampled_data_df['rounded_datetime'].dt.year.max()}."
                ),
                "The y-axis represents Proportion of major hurricane fixes to all hurricane fixes.",
                (
                    "The plot shows the proportion of major hurricane fixes over time,"
                    f" binned by {resample_period}."
                ),
            ]

            plot_line_color = params.get('line_color', '#0076D6')
            color_map = {
                '#0076D6': 'blue',
                '#707070': 'gray',
                '#d95f02': 'orange',
                '#7570b3': 'dark blue',
            }
            plot_line_color_name = color_map.get(plot_line_color, plot_line_color)
            alt_text_parts.append(
                f"The data line is plotted in {plot_line_color_name}. It has a"
                f" minimum value of {min_prop_data:.4f} at year"
                f" {min_prop_year_data}, a maximum value of {max_prop_data:.4f}"
                f" at year {max_prop_year_data}, and an average of {avg_prop_data:.4f}."
            )

            if len(trend_line) > 0:
                min_prop_trend = np.min(trend_line)
                max_prop_trend = np.max(trend_line)
                avg_prop_trend = np.mean(trend_line)
                trend_years = resampled_data_df['rounded_datetime'].dt.year.values
                min_prop_year_trend = (
                    trend_years[np.argmin(trend_line)] if len(trend_line) > 0
                    else None)
                max_prop_year_trend = (
                    trend_years[np.argmax(trend_line)] if len(trend_line) > 0
                    else None)

                alt_text_parts.append("A trend line is also shown.")
                trend_line_color = params.get('trend_line_color', '#707070')
                trend_line_color_name = color_map.get(
                    trend_line_color, trend_line_color)
                alt_text_parts.append(
                    f"The trend line is plotted in {trend_line_color_name}. It has"
                    f" a minimum value of {min_prop_trend:.4f}"
                    + (f" at year {min_prop_year_trend}"
                       if min_prop_year_trend is not None else "")
                    + f", a maximum value of {max_prop_trend:.4f}"
                    + (f" at year {max_prop_year_trend}"
                       if max_prop_year_trend is not None else "")
                    + f", and an average of {avg_prop_trend:.4f}.")

                if slope is not None and len(years_for_linregress) > 1:
                    alt_text_parts.append(
                        f"The trend line indicates a slope of {slope:.4f}.")

            alt_text_string = " ".join(alt_text_parts)
            formatted_alt_text = simple_word_wrap(alt_text_string, wrap_width=80)
            print(f"Alt text: {formatted_alt_text}")

        except Exception as manual_alt_text_e:
            print(f"Error generating alt text: {manual_alt_text_e}")
            traceback.print_exc()
            alt_text_string = (
                "Alternative text could not be generated due to"
                " an error.")
            print(f"Alt text: {alt_text_string}")

        fig_caption = (
            'Figure Caption: This plot represents a time series of the proportions of '
            'major hurricanes to all hurricanes. This proportion is calculated by the ratio: '
            'sum of storms(>= 96 kts)  /  sum of storms(>= 64 kts) or # of major hurricanes / # of hurricanes) '
            'for the ADT-HURSAT data in the North Atlantic basin. Lower proportions indicate that '
            'fewer hurricanes intensify to major hurricanes in a given year. Conversely, higher '
            'proportions indicate that a larger number of hurricanes intensify to major hurricanes '
            'in that year. The trend line in this plot is positive, indicating the percentage of '
            'hurricanes strengthening to major hurricane status has increased over time.'
        )

        plt.show()

        return {
            'filename': 'proportional_intensities_time_series.png',
            'figure': fig,
            'alt_text': alt_text_string,
            'caption' : fig_caption
        }

    except Exception as e:
        print('ERROR WITH PROPORTIONAL INTENSITIES TIMESERIES')
        traceback.print_exc()
        alt_text_string = "Alternative text could not be generated due to an error."
        print(f"Alt text: {simple_word_wrap(alt_text_string)}")
        return {
            'filename': 'proportional_intensities_time_series.png',
            'figure': None,
            'alt_text': alt_text_string
        }



def generate_wind_speed_time_series(adt_hursat_dataframe, basin, params):
    """
    Generates a time series plot of wind speeds with external parameters and returns
    the figure object, filename, and alt text. Constructs alt text.

    STATISTICAL METHOD: BINNING & AVERAGING
    1. Bins data by `resample_period` (e.g., 1 Year).
    2. Calculates the MEAN wind speed for all fixes in that bin.

    Args:
      adt_hursat_dataframe: DataFrame containing ADT-HURSAT data.
      basin: The basin name for the plot title.
      params: Dictionary of plotting parameters.

    Returns:
      A dictionary containing the figure object, filename, and alt text.
    """
    alt_text_string = "Alternative text could not be generated."
    try:
        # Create a copy to avoid modifying the original dataframe passed to the function
        adt_hursat_wind_speed_dataframe = adt_hursat_dataframe.copy()

        # Check if the input DataFrame is empty
        if adt_hursat_wind_speed_dataframe.empty:
            print(f"No data available to generate the wind speed time series for the {basin} Basin.")
            # alt text for no data
            alt_text_string = (
                f"Time series plot for ADT-HURSAT Wind Speed in the {basin} Basin:"
                " No data available for the selected parameters.")
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {
                'filename': 'wind_speed_time_series.png',
                'figure': None,
                'alt_text': alt_text_string
            }

        # Ensure the dataframe has the necessary columns before proceeding
        required_cols = ['Date', 'Time', 'WindSpeed']
        if not all(col in adt_hursat_wind_speed_dataframe.columns
                   for col in required_cols):
            print("Error: Input dataframe for Wind Speed Time Series is missing"
                  f" required columns: {', '.join(required_cols)}.")
            alt_text_string = (
                f"Time series plot for ADT-HURSAT Wind Speed in the {basin}"
                " Basin: Missing required data columns.")
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {
                'filename': 'wind_speed_time_series.png',
                'figure': None,
                'alt_text': alt_text_string
            }

        resample_period = params.get('resample_period', '1YE')
        # Check if 'rounded_datetime' exists after transformations and is datetime
        if 'rounded_datetime' not in adt_hursat_wind_speed_dataframe.columns or not pd.api.types.is_datetime64_any_dtype(
            adt_hursat_wind_speed_dataframe['rounded_datetime']):
             # If not, try to create it from 'Date' and 'Time' if available
             if all(col in adt_hursat_wind_speed_dataframe.columns for col in ['Date', 'Time']):
                  try:
                      adt_hursat_wind_speed_dataframe['datetime_str'] = (
                           adt_hursat_wind_speed_dataframe['Date'].astype(str) + ' ' +
                           adt_hursat_wind_speed_dataframe['Time'].astype(str))
                      adt_hursat_wind_speed_dataframe['rounded_datetime'] = pd.to_datetime(
                           adt_hursat_wind_speed_dataframe['datetime_str'], format='%Y%b%d %H%M%S', errors='coerce').dt.round('h')
                  except Exception as dt_conversion_e:
                       print(f"Error converting Date and Time to datetime: {dt_conversion_e}")
                       alt_text_string = (
                           "Time series plot for ADT-HURSAT Wind Speed in the"
                           f" {basin} Basin: Error processing datetime columns."
                       )
                       print(f"Alt text: {simple_word_wrap(alt_text_string)}")
                       return {'filename': 'wind_speed_time_series.png', 'figure': None, 'alt_text': alt_text_string}
             else:
                  print("Error: 'rounded_datetime' column not found, and unable to create it from 'Date' and 'Time'.")
                  alt_text_string = (
                      "Time series plot for ADT-HURSAT Wind Speed in the"
                      f" {basin} Basin: Missing required time columns."
                  )
                  print(f"Alt text: {simple_word_wrap(alt_text_string)}")
                  return {'filename': 'wind_speed_time_series.png', 'figure': None, 'alt_text': alt_text_string}

        # Set 'rounded_datetime' as index temporarily for resampling and use the to_period to convert datetime to yearly period
        # NOTE: .mean() confirms this is AVERAGING wind speeds, not just counting.
        resampled_wspd_df = adt_hursat_wind_speed_dataframe.set_index(
            'rounded_datetime')['WindSpeed'].resample(resample_period).mean().to_period('Y')

        # Check if the resampled data is empty
        if resampled_wspd_df.empty:
            print(f"No resampled named storm data available to generate the time"
                  f" series for the {basin} Basin.")
            #  construct alt text indicating no data
            alt_text_string = (
                "Time series plot for ADT-HURSAT Wind Speed in the"
                f" {basin} Basin: No resampled data available for the selected"
                " parameters.")
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {
                'filename': 'wind_speed_time_series.png',
                'figure': None,
                'alt_text': alt_text_string
            }

        # Calculate trend line using the index of the resampled data directly
        years = resampled_wspd_df.index.year
        # Only drop NaNs for the actual linregress calculation data
        trend_data_for_linregress = resampled_wspd_df.dropna()
        years_for_linregress = trend_data_for_linregress.index.year

        trend_line = []  # Initialize trend_line
        slope = None  # Initialize slope
        if len(years_for_linregress) > 1:
            try:
                slope, intercept, r_value, p_value, std_err = linregress(
                    years_for_linregress, trend_data_for_linregress.values)
                # Calculate trend line values for the *original* resampled data index
                trend_line = slope * years + intercept
            except Exception as linregress_e:
                print(f"Error calculating trend line: {linregress_e}")
                # Continue without a trend line if calculation fails

        fig, ax = plt.subplots(figsize=params.get('figsize', (14, 6)))

        #first convert the datetime (now as index) back to datetime from yearly period.
        ax.plot(
            resampled_wspd_df.index.to_timestamp(),
            resampled_wspd_df,
            label=f'{resample_period} Means', # "Means" is correct for averaging logic
            color=params.get('line_color', '#0076D6'),
            marker=params.get('data_marker', 'o'))  # Added marker parameter

        if len(trend_line) > 0:
            ax.plot(
                resampled_wspd_df.index.to_timestamp(),
                trend_line,
                linestyle=params.get('trend_line_linestyle', '--'),
                color=params.get('trend_line_color', '#707070'),
                label='Trend Line')

        ax.set_xlabel(params.get('xlabel', 'Year'))
        ax.set_ylabel(params.get('ylabel', 'WindSpeed (knots)'))
        ax.set_title(
            params.get(
                'title',
                'ADT-HURSAT Wind Speed (knots) Time Series for all fixes >= 34 kts,'
                f' {basin} Basin'))
        ax.legend(
            loc=params.get('legend_loc', 'lower center'),
            bbox_to_anchor=params.get('legend_bbox_to_anchor', (0.5, -0.3)),
            ncol=params.get('legend_ncol', 6),
            fontsize=params.get('legend_fontsize', 7))
        ax.grid(params.get('grid', True))
        plt.xticks(
            rotation=params.get('xticks_rotation', 45),
            ha=params.get('xticks_ha', 'right'),
            fontsize=params.get('xticks_fontsize', 7))
        plt.tight_layout()

        # Add minor ticks to the x-axis at 1-year intervals using YearLocator and ensure no labels
        ax.xaxis.set_minor_locator(mdates.YearLocator())
        ax.xaxis.set_minor_formatter(
            ticker.NullFormatter())  # Set minor formatter to NullFormatter

        # ---  Construct Alt Text ---\
        try:
            #use helper function to get the acronym properly formatted for alt text
            original_title = ax.get_title()
            alt_text_title = format_acronyms_for_screen_reader(original_title)

            # Get relevant data ranges and descriptive statistics for the data line
            min_wspd_data = resampled_wspd_df.min()
            max_wspd_data = resampled_wspd_df.max()
            avg_wspd_data = resampled_wspd_df.mean()
            min_wspd_year_data = resampled_wspd_df.idxmin().year  # Get year of min value
            max_wspd_year_data = resampled_wspd_df.idxmax().year  # Get year of max value

            alt_text_parts = [
                f"Time series plot titled '{alt_text_title or 'Wind Speed Time Series'}'.",
                (
                    f"The x-axis represents Year from"
                    f" {resampled_wspd_df.index.year.min()} to"
                    f" {resampled_wspd_df.index.year.max()}."
                ),
                "The y-axis represents Wind Speed in knots.",
                (
                    "The plot shows the mean wind speed over time, binned by"
                    f" {resample_period}."
                ),
            ]

            # Add details about the plot line color and statistics
            plot_line_color = params.get('line_color', '#0076D6')
            # Map common hex colors to names for better readability in alt text
            color_map = {
                '#0076D6': 'blue',
                '#707070': 'gray',
                '#d95f02': 'orange',
                '#7570b3': 'dark blue',
                # Add other color mappings as needed
            }
            plot_line_color_name = color_map.get(
                plot_line_color,
                plot_line_color)  # Use name if available, otherwise use hex
            alt_text_parts.append(
                f"The data line is plotted in {plot_line_color_name}. It has a"
                f" minimum value of {min_wspd_data:.1f} at year"
                f" {min_wspd_year_data}, a maximum value of {max_wspd_data:.1f}"
                f" at year {max_wspd_year_data}, and an average of {avg_wspd_data:.1f}."
            )

            if len(trend_line) > 0:
                # Get relevant data ranges and descriptive statistics for the trend line
                # Calculate mean correctly for the trend_line array
                min_wspd_trend = np.min(trend_line)
                max_wspd_trend = np.max(trend_line)
                avg_wspd_trend = np.mean(trend_line)
                # For the trend line, min/max will be at the start/end years
                min_wspd_year_trend = resampled_wspd_df.index.year.min(
                ) if trend_line[0] < trend_line[-1] else resampled_wspd_df.index.year.max()
                max_wspd_year_trend = resampled_wspd_df.index.year.max(
                ) if trend_line[-1] > trend_line[0] else resampled_wspd_df.index.year.min()

                alt_text_parts.append("A trend line is also shown.")
                # Add details about the trend line color and statistics
                trend_line_color = params.get('trend_line_color', '#707070')
                trend_line_color_name = color_map.get(
                    trend_line_color,
                    trend_line_color)  # Use name if available, otherwise use hex
                alt_text_parts.append(
                    f"The trend line is plotted in {trend_line_color_name}. It has"
                    f" a minimum value of {min_wspd_trend:.1f} at year"
                    f" {min_wspd_year_trend}, a maximum value of {max_wspd_trend:.1f}"
                    f" at year {max_wspd_year_trend}, and an average of {avg_wspd_trend:.1f}."
                )

                if slope is not None and len(years_for_linregress) > 1:
                    alt_text_parts.append(
                        f"The trend line indicates a slope of {slope:.2f}.")

            alt_text_string = " ".join(alt_text_parts)
            formatted_alt_text = simple_word_wrap(alt_text_string, wrap_width=80)
            print(f"Alt text: {formatted_alt_text}")

        except Exception as manual_alt_text_e:
            print(f"Error generating alt text: {manual_alt_text_e}")
            traceback.print_exc()
            alt_text_string = "Alternative text could not be generated due to an error."
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")

        fig_caption = (
            'Figure Caption: This is a time series plot of wind speeds (in knots) for all '
            'of the tropical storm fixes (>= 34 knots) in the ADT-HURSAT data for the North Atlantic Basin. '
            'This time series was created by calculating the 1-year averages of the wind speed. '
            'The trend line shows an increasing trend in wind speeds throughout the time period, '
            'indicating that over time, wind speeds for all tropical storm fixes in the ADT-HURSAT '
            'data for the North Atlantic basin have increased.'
        )

        plt.show()

        return {
            'filename': 'wind_speed_time_series.png',
            'figure': fig,
            'alt_text': alt_text_string,
            'caption' : fig_caption
        }

    except Exception as e:
        print('ERROR WITH WIND SPEED TIMESERIES')
        traceback.print_exc()
        print(e)
        alt_text_string = "Alternative text could not be generated due to an error."
        print(f"Alt text: {simple_word_wrap(alt_text_string)}")
        return {
            'filename': 'wind_speed_time_series.png',
            'figure': None,
            'alt_text': alt_text_string
        }



def generate_time_series_all_hurricanes(
    adt_hursat_dataframe, basin, params):
  """
  Generates a time series plot of wind speeds for all hurricane fixes with external
  parameters and returns the figure object, filename, and alt text.

  Constructs alt text.

  STATISTICAL METHOD: BINNING & AVERAGING
  1. Filters data for fixes >= Hurricane Threshold.
  2. Bins filtered data by `resample_period`.
  3. Calculates the MEAN wind speed for all fixes in that bin.

  Args:
    adt_hursat_dataframe: DataFrame containing ADT-HURSAT data.
    basin: The basin name for the plot title.
    params: Dictionary of plotting parameters.

  Returns:
    A dictionary containing the figure object, filename, and alt text.
  """
  alt_text_string = "Alternative text could not be generated."

  try:
    # Create a copy to avoid modifying the original dataframe.
    adt_hursat_all_hurr_dataframe = adt_hursat_dataframe.copy()

    # Check if the input DataFrame is empty.
    if adt_hursat_all_hurr_dataframe.empty:
      print(
          "No data available to generate the time series for all hurricane fixes for"
          f" the {basin} Basin."
      )
      # alt text for no data.
      alt_text_string = (
          "Time series plot for all hurricane fixes in the"
          f" {basin} Basin: No data available for the selected parameters."
      )
      print(f"Alt text: {simple_word_wrap(alt_text_string)}")
      return {
          'filename': 'all_hurricanes_time_series.png',
          'figure': None,
          'alt_text': alt_text_string,
      }

    # Ensure the dataframe has the necessary columns before proceeding.
    required_cols = ['Date', 'Time', 'WindSpeed']
    if not all(
        col in adt_hursat_all_hurr_dataframe.columns for col in required_cols
    ):
      print(
          "Error: Input dataframe for All Hurricane Fixes Time Series is missing"
          f" required columns: {', '.join(required_cols)}."
      )
      alt_text_string = (
          "Time series plot for all hurricane fixes in the"
          f" {basin} Basin: Missing required data columns."
      )
      print(f"Alt text: {simple_word_wrap(alt_text_string)}")
      return {
          'filename': 'all_hurricanes_time_series.png',
          'figure': None,
          'alt_text': alt_text_string,
      }

    hurr_threshold = 64

    # Filter for wind speeds greater than or equal to the hurricane threshold.
    # Explicitly create a copy to avoid SettingWithCopyWarning
    filtered_hurr_df = adt_hursat_all_hurr_dataframe[
        adt_hursat_all_hurr_dataframe['WindSpeed'] >= hurr_threshold
    ].copy()

    # Check if the filtered data for hurricanes is empty.
    if filtered_hurr_df.empty:
      print(
          "No data meeting the hurricane threshold available to generate the time"
          f" series for the {basin} Basin."
      )
      # alt text for no filtered data.
      alt_text_string = (
          "Time series plot for all hurricane fixes in the"
          f" {basin} Basin: No data meeting the hurricane threshold"
          f" ({hurr_threshold} kts) available."
      )
      print(f"Alt text: {simple_word_wrap(alt_text_string)}")
      return {
          'filename': 'all_hurricanes_time_series.png',
          'figure': None,
          'alt_text': alt_text_string,
      }

    resample_period = params.get('resample_period', '1YE')

    # Ensure 'rounded_datetime' exists before setting as index for resampling.
    if (
        'rounded_datetime' not in filtered_hurr_df.columns
        or not pd.api.types.is_datetime64_any_dtype(
            filtered_hurr_df['rounded_datetime']
        )
    ):
      # If not, try to create it from 'Date' and 'Time' if available.
      if all(
          col in filtered_hurr_df.columns for col in ['Date', 'Time']
      ):
        try:
          filtered_hurr_df['datetime_str'] = (
              filtered_hurr_df['Date'].astype(str)
              + ' '
              + filtered_hurr_df['Time'].astype(str)
          )
          filtered_hurr_df['rounded_datetime'] = pd.to_datetime(
              filtered_hurr_df['datetime_str'],
              format='%Y%b%d %H%M%S',
              errors='coerce',
          ).dt.round('h')
        except Exception as dt_conversion_e:
          print(f"Error converting Date and Time to datetime: {dt_conversion_e}")
          alt_text_string = (
              "Time series plot for all hurricane fixes in the"
              f" {basin} Basin: Error processing datetime columns."
          )
          print(f"Alt text: {simple_word_wrap(alt_text_string)}")
          return {
              'filename': 'all_hurricanes_time_series.png',
              'figure': None,
              'alt_text': alt_text_string,
          }
      else:
        print(
            "Error: 'rounded_datetime' column not found, and unable to create"
            " it from 'Date' and 'Time'."
        )
        alt_text_string = (
            "Time series plot for all hurricane fixes in the"
            f" {basin} Basin: Missing required time columns."
        )
        print(f"Alt text: {simple_word_wrap(alt_text_string)}")
        return {
            'filename': 'all_hurricanes_time_series.png',
            'figure': None,
            'alt_text': alt_text_string,
        }


    # Resample the WindSpeed data by the specified period and calculate the mean.
    # NOTE: .mean() confirms this is AVERAGING wind speeds, not just counting.
    resampled_hurr_wspd_df = (
        filtered_hurr_df.set_index('rounded_datetime')['WindSpeed']
        .resample(resample_period)
        .mean().to_period('Y')
    )

    # Check if the resampled hurricane data is empty.
    if resampled_hurr_wspd_df.empty:
      print(
          "No resampled hurricane fix data available to generate the time series for"
          f" the {basin} Basin."
      )
      # alt text for no resampled data.
      alt_text_string = (
          "Time series plot for all hurricane fixes in the"
          f" {basin} Basin: No resampled hurricane data available for the"
          " selected period."
      )
      print(f"Alt text: {simple_word_wrap(alt_text_string)}")
      return {
          'filename': 'all_hurricanes_time_series.png',
          'figure': None,
          'alt_text': alt_text_string,
      }

    # Calculate trend line using the index of the resampled data.
    years = resampled_hurr_wspd_df.index.year
    # Only drop NaNs for the actual linregress calculation data.
    trend_data_for_linregress = resampled_hurr_wspd_df.dropna()
    years_for_linregress = trend_data_for_linregress.index.year

    trend_line = []  # Initialize trend_line.
    slope = None  # Initialize slope.
    if len(years_for_linregress) > 1:
      try:
        slope, intercept, r_value, p_value, std_err = linregress(
            years_for_linregress, trend_data_for_linregress.values
        )
        # Calculate trend line values for the *original* resampled data index.
        trend_line = slope * years + intercept
      except Exception as linregress_e:
        print(f"Error calculating trend line: {linregress_e}")
        # Continue without a trend line if calculation fails.

    # Create the figure and axes for the plot.
    fig, ax = plt.subplots(figsize=params.get('figsize', (14, 6)))

    # Plot the resampled wind speed data.
    ax.plot(
        resampled_hurr_wspd_df.index.to_timestamp(),
        resampled_hurr_wspd_df,
        label=f'{resample_period} Means', # "Means" is correct for averaging logic
        color=params.get('line_color', '#0076D6'),
        marker=params.get('data_marker', 'o'),
    )

    # Plot the trend line if calculated.
    if len(trend_line) > 0:
      ax.plot(
          resampled_hurr_wspd_df.index.to_timestamp(),
          trend_line,
          linestyle=params.get('trend_line_linestyle', '--'),
          color=params.get('trend_line_color', '#707070'),
          label='Trend Line',
      )

    # Set plot labels and title.
    ax.set_xlabel(params.get('xlabel', 'Year'))
    ax.set_ylabel(params.get('ylabel', 'WindSpeed (knots)'))
    ax.set_title(
        params.get(
            'title',
            'ADT-HURSAT Wind Speed (knots) Time Series for All Hurricane Fixes'
            f' (>= {hurr_threshold} kts), {basin} Basin',
        )
    )
    ax.legend(
        loc=params.get('legend_loc', 'lower center'),
        bbox_to_anchor=params.get('legend_bbox_to_anchor', (0.5, -0.3)),
        ncol=params.get('legend_ncol', 6),
        fontsize=params.get('legend_fontsize', 7),
    )
    ax.grid(params.get('grid', True))
    plt.xticks(
        rotation=params.get('xticks_rotation', 45),
        ha=params.get('xticks_ha', 'right'),
        fontsize=params.get('xticks_fontsize', 7),
    )
    plt.tight_layout()

    # Add minor ticks to the x-axis at 1-year intervals and ensure no labels.
    ax.xaxis.set_minor_locator(mdates.YearLocator())
    ax.xaxis.set_minor_formatter(
        ticker.NullFormatter()
    )  # Set minor formatter to NullFormatter

    # --- Construct Alt Text ---
    try:
      #use helper function to get the acronym properly formatted for alt text
      original_title = ax.get_title()
      alt_text_title = format_acronyms_for_screen_reader(original_title)

      # Get relevant data ranges and descriptive statistics for the data line.
      min_wspd_data = resampled_hurr_wspd_df.min()
      max_wspd_data = resampled_hurr_wspd_df.max()
      avg_wspd_data = resampled_hurr_wspd_df.mean()
      # Get year of min/max value.
      min_wspd_year_data = resampled_hurr_wspd_df.idxmin().year
      max_wspd_year_data = resampled_hurr_wspd_df.idxmax().year

      alt_text_parts = [
          f"Time series plot titled '{alt_text_title or 'All Hurricane Fixes Wind Speed Time Series'}'.",
          (
              f"The x-axis represents Year from"
              f" {resampled_hurr_wspd_df.index.year.min()} to"
              f" {resampled_hurr_wspd_df.index.year.max()}."
          ),
          "The y-axis represents Wind Speed in knots.",
          (
              "The plot shows the mean wind speed over time for hurricane fixes"
              f" (>= {hurr_threshold} kts), binned by {resample_period}."
          ),
      ]

      # Add details about the plot line color and statistics.
      plot_line_color = params.get('line_color', '#0076D6')
      # Map common hex colors to names for better readability in alt text.
      color_map = {
          '#0076D6': 'blue',
          '#707070': 'gray',
          '#d95f02': 'orange',
          '#7570b3': 'dark blue',
          # Add other color mappings as needed.
      }
      plot_line_color_name = color_map.get(
          plot_line_color, plot_line_color
      )  # Use name if available, otherwise use hex.
      alt_text_parts.append(
          f"The data line is plotted in {plot_line_color_name}. It has a"
          f" minimum value of {min_wspd_data:.1f} at year"
          f" {min_wspd_year_data}, a maximum value of {max_wspd_data:.1f} at year"
          f" {max_wspd_year_data}, and an average of {avg_wspd_data:.1f}."
      )

      if len(trend_line) > 0:
        # Get relevant data ranges and descriptive statistics for the trend line.
        min_wspd_trend = np.min(trend_line)
        max_wspd_trend = np.max(trend_line)
        avg_wspd_trend = np.mean(trend_line)
        # For the trend line, min/max will be at the start/end years.
        min_wspd_year_trend = (
            resampled_hurr_wspd_df.index.year.min()
            if trend_line[0] < trend_line[-1]
            else resampled_hurr_wspd_df.index.year.max()
        )
        max_wspd_year_trend = (
            resampled_hurr_wspd_df.index.year.max()
            if trend_line[-1] > trend_line[0]
            else resampled_hurr_wspd_df.index.year.min()
        )

        alt_text_parts.append("A trend line is also shown.")
        # Add details about the trend line color and statistics.
        trend_line_color = params.get('trend_line_color', '#707070')
        trend_line_color_name = color_map.get(
            trend_line_color, trend_line_color
        )  # Use name if available, otherwise use hex.
        alt_text_parts.append(
            f"The trend line is plotted in {trend_line_color_name}. It has"
            f" a minimum value of {min_wspd_trend:.1f} at year"
            f" {min_wspd_year_trend}, a maximum value of {max_wspd_trend:.1f}"
            f" at year {max_wspd_year_trend}, and an average of"
            f" {avg_wspd_trend:.1f}."
        )

        if slope is not None and len(years_for_linregress) > 1:
          alt_text_parts.append(
              f"The trend line indicates a slope of {slope:.2f}."
          )

      alt_text_string = " ".join(alt_text_parts)
      formatted_alt_text = simple_word_wrap(alt_text_string, wrap_width=80)
      print(f"Alt text: {formatted_alt_text}")

    except Exception as manual_alt_text_e:
      print(f"Error generating alt text: {manual_alt_text_e}")
      traceback.print_exc()
      alt_text_string = "Alternative text could not be generated due to an error."

    fig_caption = (
        'Figure Caption: This is a time series plot of wind speeds (in knots) for '
        'all hurricane fixes (>= 64 knots) in the ADT-HURSAT data for '
        'the North Atlantic Basin. This time series was created by calculating the 1-year averages '
        'of the wind speed. The trend line shows an increasing trend in wind speeds '
        'throughout the time period, indicating that wind speeds for all hurricane fixes in the '
        'ADT-HURSAT data set have increased over time.'
    )

    plt.show()

    return {
        'filename': 'all_hurricanes_time_series.png',
        'figure': fig,
        'alt_text': alt_text_string,
        'caption' : fig_caption
    }

  except Exception as e:
    print('ERROR WITH ALL HURRICANE FIXES TIMESERIES')
    traceback.print_exc()
    print(e)
    return None



def generate_time_series_major_hurricanes(
    adt_hursat_dataframe, basin, params):
  """
  Generates a time series plot of wind speeds for major hurricane fixes.

  Uses external parameters for customization and returns the figure object,
  filename, and constructed alt text.

  STATISTICAL METHOD: BINNING & AVERAGING
  1. Filters data for fixes >= Major Hurricane Threshold.
  2. Bins filtered data by `resample_period`.
  3. Calculates the MEAN wind speed for all fixes in that bin.

  Args:
    adt_hursat_dataframe: DataFrame containing ADT-HURSAT data.
    basin: The basin name for the plot title.
    params: Dictionary of plotting parameters.

  Returns:
    A dictionary containing the figure object, filename, and alt text.
  """
  alt_text_string = "Alternative text could not be generated."

  try:
    # Create a copy to avoid modifying the original dataframe.
    adt_hursat_major_hurr_dataframe = adt_hursat_dataframe.copy()

    # Check if the input DataFrame is empty.
    if adt_hursat_major_hurr_dataframe.empty:
      print(
          "No data available to generate the time series for major hurricane fixes"
          f" for the {basin} Basin."
      )
      # alt text for no data.
      alt_text_string = (
          "Time series plot for major hurricane fixes in the"
          f" {basin} Basin: No data available for the selected parameters."
      )
      print(f"Alt text: {simple_word_wrap(alt_text_string)}")
      return {
          'filename': 'major_hurricanes_time_series.png',
          'figure': None,
          'alt_text': alt_text_string,
      }

    # Ensure the dataframe has the necessary columns before proceeding.
    required_cols = ['Date', 'Time', 'WindSpeed']
    if not all(
        col in adt_hursat_major_hurr_dataframe.columns for col in required_cols
    ):
      print(
          "Error: Input dataframe for Major Hurricane Fixes Time Series is missing"
          f" required columns: {', '.join(required_cols)}."
      )
      alt_text_string = (
          "Time series plot for major hurricane fixes in the"
          f" {basin} Basin: Missing required data columns."
      )
      print(f"Alt text: {simple_word_wrap(alt_text_string)}")
      return {
          'filename': 'major_hurricanes_time_series.png',
          'figure': None,
          'alt_text': alt_text_string,
      }

    major_threshold = 96

    # Filter for wind speeds greater than or equal to the major hurricane
    # threshold.
    # Explicitly create a copy to avoid SettingWithCopyWarning
    filtered_resampled_major_hurr_df = adt_hursat_major_hurr_dataframe[
        adt_hursat_major_hurr_dataframe['WindSpeed'] >= major_threshold
    ].copy()

    # Check if the filtered data for major hurricanes is empty.
    if filtered_resampled_major_hurr_df.empty:
      print(
          "No data meeting the major hurricane threshold available to generate"
          f" the time series for the {basin} Basin."
      )
      # alt text for no filtered data.
      alt_text_string = (
          "Time series plot for major hurricane fixes in the"
          f" {basin} Basin: No data meeting the major hurricane threshold"
          f" ({major_threshold} kts) available."
      )
      print(f"Alt text: {simple_word_wrap(alt_text_string)}")
      return {
          'filename': 'major_hurricanes_time_series.png',
          'figure': None,
          'alt_text': alt_text_string,
      }

    resample_period = params.get('resample_period', '1YE')

    # Ensure 'rounded_datetime' exists before setting as index for resampling.
    if (
        'rounded_datetime' not in filtered_resampled_major_hurr_df.columns
        or not pd.api.types.is_datetime64_any_dtype(
            filtered_resampled_major_hurr_df['rounded_datetime']
        )
    ):
      # If not, try to create it from 'Date' and 'Time' if available.
      if all(
          col in filtered_resampled_major_hurr_df.columns
          for col in ['Date', 'Time']
      ):
        try:
          filtered_resampled_major_hurr_df['datetime_str'] = (
              filtered_resampled_major_hurr_df['Date'].astype(str)
              + ' '
              + filtered_resampled_major_hurr_df['Time'].astype(str)
          )
          filtered_resampled_major_hurr_df['rounded_datetime'] = pd.to_datetime(
              filtered_resampled_major_hurr_df['datetime_str'],
              format='%Y%b%d %H%M%S',
              errors='coerce',
          ).dt.round('h')
        except Exception as dt_conversion_e:
          print(f"Error converting Date and Time to datetime: {dt_conversion_e}")
          alt_text_string = (
              "Time series plot for major hurricane fixes in the"
              f" {basin} Basin: Error processing datetime columns."
          )
          print(f"Alt text: {simple_word_wrap(alt_text_string)}")
          return {
              'filename': 'major_hurricanes_time_series.png',
              'figure': None,
              'alt_text': alt_text_string,
          }
      else:
        print(
            "Error: 'rounded_datetime' column not found, and unable to create"
            " it from 'Date' and 'Time'."
        )
        alt_text_string = (
            "Time series plot for major hurricane fixes in the"
            f" {basin} Basin: Missing required time columns."
        )
        print(f"Alt text: {simple_word_wrap(alt_text_string)}")
        return {
            'filename': 'major_hurricanes_time_series.png',
            'figure': None,
            'alt_text': alt_text_string,
        }

    # Resample the WindSpeed data by the specified period and calculate the mean.
    # NOTE: .mean() confirms this is AVERAGING wind speeds, not just counting.
    resampled_major_hurr_wspd_df = (
        filtered_resampled_major_hurr_df.set_index('rounded_datetime')[
            'WindSpeed'
        ].resample(resample_period).mean().to_period('Y')
    )

    # Check if the resampled major hurricane data is empty.
    if resampled_major_hurr_wspd_df.empty:
      print(
          "No resampled major hurricane fix data available to generate the time"
          f" series for the {basin} Basin."
      )
      # alt text for no resampled data.
      alt_text_string = (
          "Time series plot for major hurricane fixes in the"
          f" {basin} Basin: No resampled major hurricane data available for the"
          " selected period."
      )
      print(f"Alt text: {simple_word_wrap(alt_text_string)}")
      return {
          'filename': 'major_hurricanes_time_series.png',
          'figure': None,
          'alt_text': alt_text_string,
      }

    # Calculate trend line using the index of the resampled data.
    years = resampled_major_hurr_wspd_df.index.year
    # Only drop NaNs for the actual linregress calculation data.
    trend_data_for_linregress = resampled_major_hurr_wspd_df.dropna()
    years_for_linregress = trend_data_for_linregress.index.year

    trend_line = []  # Initialize trend_line.
    slope = None  # Initialize slope.
    if len(years_for_linregress) > 1:
      try:
        slope, intercept, r_value, p_value, std_err = linregress(
            years_for_linregress, trend_data_for_linregress.values
        )
        # Calculate trend line values for the *original* resampled data index.
        trend_line = slope * years + intercept
      except Exception as linregress_e:
        print(f"Error calculating trend line: {linregress_e}")
        # Continue without a trend line if calculation fails.

    # Create the figure and axes for the plot.
    fig, ax = plt.subplots(figsize=params.get('figsize', (14, 6)))

    # Plot the resampled major hurricane wind speed data.
    ax.plot(
        resampled_major_hurr_wspd_df.index.to_timestamp(),
        resampled_major_hurr_wspd_df,
        label=f'{resample_period} Means', # "Means" is correct for averaging logic
        color=params.get('line_color', '#0076D6'),
        marker=params.get('data_marker', 'o'),
    )

    # Plot the trend line if calculated.
    if len(trend_line) > 0:
      ax.plot(
          resampled_major_hurr_wspd_df.index.to_timestamp(),
          trend_line,
          linestyle=params.get('trend_line_linestyle', '--'),
          color=params.get('trend_line_color', '#707070'),
          label='Trend Line',
      )

    # Set plot labels and title.
    ax.set_xlabel(params.get('xlabel', 'Year'))
    ax.set_ylabel(params.get('ylabel', 'WindSpeed (knots)'))
    ax.set_title(
        params.get(
            'title',
            'ADT-HURSAT Wind Speed (knots) Time Series for Major Hurricane Fixes'
            f' (>= {major_threshold} kts), {basin} Basin',
        )
    )
    ax.legend(
        loc=params.get('legend_loc', 'lower center'),
        bbox_to_anchor=params.get('legend_bbox_to_anchor', (0.5, -0.3)),
        ncol=params.get('legend_ncol', 6),
        fontsize=params.get('legend_fontsize', 7),
    )
    ax.grid(params.get('grid', True))
    plt.xticks(
        rotation=params.get('xticks_rotation', 45),
        ha=params.get('xticks_ha', 'right'),
        fontsize=params.get('xticks_fontsize', 7),
    )
    plt.tight_layout()

    # Add minor ticks to the x-axis at 1-year intervals and ensure no labels.
    ax.xaxis.set_minor_locator(mdates.YearLocator())
    ax.xaxis.set_minor_formatter(
        ticker.NullFormatter()
    )  # Set minor formatter to NullFormatter

    # --- Construct Alt Text ---
    try:
      #use helper function to get the acronym properly formatted for alt text
      original_title = ax.get_title()
      alt_text_title = format_acronyms_for_screen_reader(original_title)

      # Get relevant data ranges and descriptive statistics for the data line.
      min_wspd_data = resampled_major_hurr_wspd_df.min()
      max_wspd_data = resampled_major_hurr_wspd_df.max()
      avg_wspd_data = resampled_major_hurr_wspd_df.mean()
      # Get year of min/max value.
      min_wspd_year_data = resampled_major_hurr_wspd_df.idxmin().year
      max_wspd_year_data = resampled_major_hurr_wspd_df.idxmax().year

      alt_text_parts = [
          "Time series plot titled"
          f" '{alt_text_title or 'Major Hurricane Fixes Wind Speed Time Series'}'.",
          (
              f"The x-axis represents Year from"
              f" {resampled_major_hurr_wspd_df.index.year.min()} to"
              f" {resampled_major_hurr_wspd_df.index.year.max()}."
          ),
          "The y-axis represents Wind Speed in knots.",
          (
              "The plot shows the mean wind speed over time for major hurricane fixes"
              f" (>= {major_threshold} kts), binned by {resample_period}."
          ),
      ]

      # Add details about the plot line color and statistics.
      plot_line_color = params.get('line_color', '#0076D6')
      # Map common hex colors to names for better readability in alt text.
      color_map = {
          '#0076D6': 'blue',
          '#707070': 'gray',
          '#d95f02': 'orange',
          '#7570b3': 'dark blue',
          # Add other color mappings as needed.
      }
      plot_line_color_name = color_map.get(
          plot_line_color, plot_line_color
      )  # Use name if available, otherwise use hex.
      alt_text_parts.append(
          f"The data line is plotted in {plot_line_color_name}. It has a"
          f" minimum value of {min_wspd_data:.1f} at year"
          f" {min_wspd_year_data}, a maximum value of {max_wspd_data:.1f} at year"
          f" {max_wspd_year_data}, and an average of {avg_wspd_data:.1f}."
      )

      if len(trend_line) > 0:
        # Get relevant data ranges and descriptive statistics for the trend line.
        min_wspd_trend = np.min(trend_line)
        max_wspd_trend = np.max(trend_line)
        avg_wspd_trend = np.mean(trend_line)
        # For the trend line, min/max will be at the start/end years.
        min_wspd_year_trend = (
            resampled_major_hurr_wspd_df.index.year.min()
            if trend_line[0] < trend_line[-1]
            else resampled_major_hurr_wspd_df.index.year.max()
        )
        max_wspd_year_trend = (
            resampled_major_hurr_wspd_df.index.year.max()
            if trend_line[-1] > trend_line[0]
            else resampled_major_hurr_wspd_df.index.year.min()
        )

        alt_text_parts.append("A trend line is also shown.")
        # Add details about the trend line color and statistics.
        trend_line_color = params.get('trend_line_color', '#707070')
        trend_line_color_name = color_map.get(
            trend_line_color, trend_line_color
        )  # Use name if available, otherwise use hex.
        alt_text_parts.append(
            f"The trend line is plotted in {trend_line_color_name}. It has"
            f" a minimum value of {min_wspd_trend:.1f} at year"
            f" {min_wspd_year_trend}, a maximum value of {max_wspd_trend:.1f}"
            f" at year {max_wspd_year_trend}, and an average of"
            f" {avg_wspd_trend:.1f}."
        )

        if slope is not None and len(years_for_linregress) > 1:
          alt_text_parts.append(
              f"The trend line indicates a slope of {slope:.2f}."
          )

      alt_text_string = " ".join(alt_text_parts)
      formatted_alt_text = simple_word_wrap(alt_text_string, wrap_width=80)
      print(f"Alt text: {formatted_alt_text}")

    except Exception as manual_alt_text_e:
      print(f"Error generating alt text: {manual_alt_text_e}")
      traceback.print_exc()
      alt_text_string = "Alternative text could not be generated due to an error."

    fig_caption = (
        'Figure Caption: This is a time series plot of wind speeds (in knots) '
        'for all major hurricane fixes (>= 96 knots) in the ADT-HURSAT data '
        'for the North Atlantic Basin. This time series was created by calculating '
        'the 1-year averages of the wind speed. Some years did not have major hurricanes, '
        'so those years are not represented on the figure. The trend line shows an increasing trend in '
        'wind speeds for major hurricanes fixes throughout the time period, indicating that wind speeds '
        'for major hurricanes fixes in the ADT-HURSAT data have increased over time.'
    )

    plt.show()

    return {
        'filename': 'major_hurricanes_time_series.png',
        'figure': fig,
        'alt_text': alt_text_string,
        'caption' : fig_caption
    }

  except Exception as e:
    print('ERROR WITH MAJOR HURRICANES TIMESERIES')
    traceback.print_exc()
    print(e)
    return None



def generate_density_norm_histo(
    adt_hursat_dataframe, basin, params):
    """
    Generates a density normalized histogram plot of wind speed.

    Constructs alt text.

    Args:
        adt_hursat_dataframe: DataFrame containing ADT-HURSAT data.
        basin: The basin name for the plot title.
        params: Dictionary of plotting parameters.

    Returns:
        A dictionary containing the figure object, filename, and alt text.
    """
    alt_text_string = "Alternative text could not be generated."
    try:
        # Create a copy of the input dataframe
        adt_hursat_histo_dataframe = adt_hursat_dataframe.copy()

        # Check if the input DataFrame is empty
        if adt_hursat_histo_dataframe.empty:
             print(f"No data available to generate the density normalized histogram for the {basin} Basin.")
             alt_text_string = (
                 f"Density normalized histogram for the {basin} Basin: No data"
                 " available for the selected parameters."
             )
             print(f"Alt text: {simple_word_wrap(alt_text_string)}")
             return {
                 'filename': 'wind_speed_density_histogram.png',
                 'figure': None,
                 'alt_text': alt_text_string
             }

        # Ensure the dataframe has the necessary columns before proceeding
        required_cols = ['WindSpeed']
        if not all(col in adt_hursat_histo_dataframe.columns
                   for col in required_cols):
            print("Error: Input dataframe for Density Normalized Histogram is"
                  f" missing required columns: {', '.join(required_cols)}.")
            alt_text_string = (
                f"Density normalized histogram for the {basin} Basin: Missing"
                " required data columns."
            )
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {
                'filename': 'wind_speed_density_histogram.png',
                'figure': None,
                'alt_text': alt_text_string
            }

        wind_speed_data = adt_hursat_histo_dataframe['WindSpeed'].dropna()

        # Check if there is enough data to plot a histogram
        if wind_speed_data.empty:
             print(f"No valid wind speed data available to generate the density normalized histogram for the {basin} Basin.")
             alt_text_string = (
                 f"Density normalized histogram for the {basin} Basin: No valid"
                 " wind speed data available."
             )
             print(f"Alt text: {simple_word_wrap(alt_text_string)}")
             return {
                 'filename': 'wind_speed_density_histogram.png',
                 'figure': None,
                 'alt_text': alt_text_string
             }

        ax = plt.gca() # Get current axes if they exist, otherwise create new ones implicitly
        if not ax: # If no current axes, create a figure and axes
             fig, ax = plt.subplots(figsize=params.get('figsize', (10, 6)))
        else: # If current axes exist, use them and potentially adjust figure size
             fig = ax.get_figure()
             fig.set_size_inches(params.get('figsize', (10, 6)))


        ax.hist(wind_speed_data,
                bins=params.get('bins', 20),
                density=params.get('density', True),
                color=params.get('color', '#084fc9'),
                alpha=params.get('alpha', 0.5))

        ax.set_title(params.get('title', f'ADT-HURSAT Histogram of Wind Speed (Density Normalized), {basin} Basin'))
        ax.set_xlabel(params.get('xlabel', 'Wind Speed (knots)'))
        ax.set_ylabel(params.get('ylabel', 'Density'))
        ax.grid(params.get('grid', True))

        plt.tight_layout()

        # --- Construct Alt Text ---
        try:
            #use helper function to get the acronym properly formatted for alt text
            original_title = ax.get_title()
            alt_text_title = format_acronyms_for_screen_reader(original_title)

            # Get relevant descriptive statistics for the histogram data
            mean_wspd = wind_speed_data.mean()
            median_wspd = wind_speed_data.median()
            std_wspd = wind_speed_data.std()
            min_wspd = wind_speed_data.min()
            max_wspd = wind_speed_data.max()
            num_data_points = len(wind_speed_data)
            num_bins = params.get('bins', 20)

            alt_text_parts = [
                f"Density normalized histogram titled '{alt_text_title or 'Wind Speed Density Histogram'}'.",
                (
                    f"The x-axis represents Wind Speed in knots, ranging from"
                    f" approximately {ax.get_xlim()[0]:.1f} to {ax.get_xlim()[1]:.1f}."
                ), # Use plot limits for range
                f"The y-axis represents Density.",
                (
                    "The histogram shows the distribution of wind speed for"
                    f" {num_data_points} data points, using {num_bins} bins."
                ),
                (
                    f"The data has a mean of {mean_wspd:.1f} knots, a median of"
                    f" {median_wspd:.1f} knots, and a standard deviation of"
                    f" {std_wspd:.1f} knots."
                ),
                f"The minimum wind speed is {min_wspd:.1f} knots and the maximum is {max_wspd:.1f} knots."
            ]

            # Add details about the histogram color
            histo_color = params.get('color', '#084fc9')
            # Map common hex colors to names
            color_map = {
                '#084fc9': 'blue',
                '#d95f02': 'orange',
                '#7570b3': 'dark blue',
                '#1b9e77': 'green',
                '#e7298a': 'pink',
                '#a6761d': 'brown',
                '#666666': 'dark gray',
                # Add other color mappings as needed
            }
            histo_color_name = color_map.get(histo_color, histo_color) # Use name if available, otherwise use hex
            alt_text_parts.append(f"The histogram bars are plotted in {histo_color_name}.")

            alt_text_string = " ".join(alt_text_parts)
            formatted_alt_text = simple_word_wrap(alt_text_string, wrap_width=80)
            print(f"Alt text: {formatted_alt_text}")

        except Exception as manual_alt_text_e:
            print(f"Error generating alt text: {manual_alt_text_e}")
            traceback.print_exc()
            alt_text_string = "Alternative text could not be generated due to an error."
            print(f"Alt text: {alt_text_string}")

        fig_caption = (
            'Figure Caption: This plot is a frequency distribution of the density '
            '(i.e., probability or relative frequency) distribution of wind speed for all '
            'storm fixes across 1978 - 2024 in the ADT-HURSAT data. This figure illustrates '
            'the overall distribution of storm intensities, helping to identify the most common '
            'intensity ranges (modes) in the dataset. The shape of the distribution is positively skewed, '
            'indicating that there is a higher probability of having a larger number of storm fixes with lower '
            'wind speeds than higher wind speeds.'
        )

        plt.show()

        return {'filename': 'wind_speed_density_histogram.png',
                'figure': fig,
                'alt_text': alt_text_string,
                'caption' : fig_caption
                }

    except Exception as e:
        print('ERROR WITH DENSITY NORM HISTOGRAM')
        traceback.print_exc()
        print(e)
        alt_text_string = "Alternative text could not be generated due to an error."
        print(f"Alt text: {simple_word_wrap(alt_text_string)}")
        return {'filename': 'wind_speed_density_histogram.png', 'figure': None, 'alt_text': alt_text_string}



def generate_intensity_distribution_plot(merged_allrecs_ibtracs_adthursat_df, basin, params):
    """
    Generates a histogram comparing the intensity distribution of ADT-HURSAT and IBTrACS data
    with external parameters and returns the figure object and filename.
    Bins are calculated based on a defined 'bin_interval' from the parameters.

    Args:
        merged_allrecs_ibtracs_adthursat_df: Merged DataFrame of ADT-HURSAT and IBTrACS data.
        basin: The basin name for the plot title.
        params: Dictionary of plotting parameters.

    Returns:
        A dictionary containing the figure object and filename.
    """
    alt_text_string = "Alternative text could not be generated."

    try:
        # Check if the input DataFrame is empty
        if merged_allrecs_ibtracs_adthursat_df.empty:
            print(f"No data available to generate the intensity distribution plot for the {basin} Basin.")
            alt_text_string = (
                f"Histogram plot for the {basin} Basin: No data available for the selected parameters."
            )
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {'filename': 'intensity_distribution_comparison.png', 'figure': None, 'alt_text': alt_text_string}

        # Ensure the dataframe has the necessary columns before proceeding
        required_cols = ['WindSpeed_adt_hursat', 'WindSpeed_ibtracs']
        if not all(col in merged_allrecs_ibtracs_adthursat_df.columns for col in required_cols):
            print(f"Error: Input dataframe for Intensity Distribution Plot is missing "
                  f"required columns: {', '.join(required_cols)}.")
            alt_text_string = (
                f"Histogram plot for the {basin} Basin: Missing required data columns."
            )
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {'filename': 'intensity_distribution_comparison.png', 'figure': None, 'alt_text': alt_text_string}

        fig, ax = plt.subplots(figsize=params.get('figsize', (10, 6)))

        adt_hursat_data = merged_allrecs_ibtracs_adthursat_df['WindSpeed_adt_hursat'].dropna()
        ibtracs_data = merged_allrecs_ibtracs_adthursat_df['WindSpeed_ibtracs'].dropna()

        bin_interval = params.get('bin_interval', 5) # Default to 5 knots interval
        # bin_interval should be knots or a multiple of 5 knots due to the format
        # of IBTrACS data

        # Combine data to find the overall min/max
        combined_data = pd.concat([adt_hursat_data, ibtracs_data])
        # Find minimum and maximum values, rounding down/up to the nearest interval
        data_min = np.floor(combined_data.min() / bin_interval) * bin_interval
        data_max = np.ceil(combined_data.max() / bin_interval) * bin_interval

        # Use numpy.arange to create fixed-interval bins
        bins_to_use = np.arange(data_min, data_max + bin_interval, bin_interval)

        # Initialize variables to hold histogram counts
        adt_counts = []
        ibtracs_counts = []

        if not adt_hursat_data.empty:
            # Capture the counts and bins from the histogram plot
            adt_counts, adt_bins, _ = ax.hist(adt_hursat_data,
                    bins=bins_to_use, # <- REFRACTORED LINE
                    alpha=params.get('adt_hursat_alpha', 0.7),
                    label=params.get('adt_hursat_label', 'ADT-HURSAT'),
                    color=params.get('adt_hursat_color', '#d95f02'))
        else:
            print("No non-NaN values in 'WindSpeed_adt_hursat' for the selected basin. "
                  f"Skipping ADT-HURSAT histogram.")

        if not ibtracs_data.empty:
            # Capture the counts and bins from the histogram plot
            ibtracs_counts, ibtracs_bins, _ = ax.hist(ibtracs_data,
                    bins=bins_to_use, # <- REFRACTORED LINE
                    alpha=params.get('ibtracs_alpha', 0.7),
                    label=params.get('ibtracs_label', 'IBTrACS'),
                    color=params.get('ibtracs_color', '#7570b3'))
        else:
            print("No non-NaN values in 'WindSpeed_ibtracs' for the selected basin. "
                  f"Skipping IBTrACS histogram.")

        ax.set_title(params.get('title', f'Distribution of Tropical Cyclone Intensity '
                                          f'for entire period, {basin} Basin'))
        ax.set_xlabel(params.get('xlabel', 'Intensity (knots)'))
        ax.set_ylabel(params.get('ylabel', 'Frequency'))
        ax.legend()
        ax.grid(axis='y', linestyle='--', alpha=0.6) # Added grid for clarity

        # Set x-axis ticks based on bin_interval for better readability
        if bin_interval > 0:
             # If bin_interval is small (e.g., 5), show every 2nd interval (10)
             # If bin_interval is larger (e.g., 10), show every 2nd interval (20)
             tick_interval = bin_interval * 2 if bin_interval < 20 else bin_interval
             ax.xaxis.set_major_locator(ticker.MultipleLocator(tick_interval))

        # --- Construct Alt Text ---
        try:
            #use helper function to get the acronym properly formatted for alt text
            original_title = ax.get_title()
            alt_text_title = format_acronyms_for_screen_reader(original_title)

            # Map common hex colors to names for better readability in alt text
            color_map = {
                '#d95f02': 'orange',
                '#7570b3': 'dark blue',
                '#0076D6': 'blue',
                '#707070': 'gray',
            }

            alt_text_parts = [
                f"Histogram plot titled '{alt_text_title}'.",
                f"The x-axis represents {ax.get_xlabel()} from approximately"
                f" {data_min:.0f} to {data_max:.0f}, with a bin interval of {bin_interval} knots."
            ]

            # Describe the Y-axis range
            y_axis_description = f"The y-axis represents {ax.get_ylabel()}."
            if len(adt_counts) > 0 or len(ibtracs_counts) > 0:
                overall_max_freq = max(adt_counts.max() if len(adt_counts) > 0 else 0,
                                       ibtracs_counts.max() if len(ibtracs_counts) > 0 else 0)
                y_axis_description += (f" The frequency, or count of observations per bin,"
                                       f" ranges up to approximately {int(overall_max_freq)}.")
            alt_text_parts.append(y_axis_description)

            # Describe the ADT-HURSAT data if it exists
            if not adt_hursat_data.empty:
                adt_color_hex = params.get('adt_hursat_color', '#d95f02')
                adt_color_name = color_map.get(adt_color_hex, adt_color_hex)
                alt_text_parts.append(
                    f"{format_acronyms_for_screen_reader('One dataset, labeled ADT-HURSAT')} and colored {adt_color_name},"
                    f" shows a distribution with an average intensity of {adt_hursat_data.mean():.1f} knots"
                    f" and a peak frequency of {int(adt_counts.max())} observations in a single intensity bin."
                )

            # Describe the IBTrACS data if it exists
            if not ibtracs_data.empty:
                ibtracs_color_hex = params.get('ibtracs_color', '#7570b3')
                ibtracs_color_name = color_map.get(ibtracs_color_hex, ibtracs_color_hex)
                alt_text_parts.append(
                    f"{format_acronyms_for_screen_reader('A second dataset, labeled IBTrACS')} and colored {ibtracs_color_name},"
                    f" shows a distribution with an average intensity of {ibtracs_data.mean():.1f} knots"
                    f" and a peak frequency of {int(ibtracs_counts.max())} observations in a single intensity bin."
                )

            alt_text_string = " ".join(alt_text_parts)
            formatted_alt_text = simple_word_wrap(alt_text_string, wrap_width=80)
            print(f"Alt text: {formatted_alt_text}")

        except Exception as alt_text_e:
            print(f"Error generating alt text: {alt_text_e}")
            traceback.print_exc()
            alt_text_string = "Alternative text could not be generated due to an error."
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")

        fig_caption = (
            'Figure Caption: This plot compares the distributions of wind speeds for all track fixes in the '
            'ADT-HURSAT and IBTrACS data, for the period 1978-2024. Both ADT-HURSAT and '
            'IBTrACS data distributions are positively skewed, indicating that there are more storms '
            'with lower intensities than higher intensities, which is expected. The ADT-HURSAT data '
            'estimates a substantially larger proportion of tropical storms (<64 knots) to minor '
            'hurricanes (64-96 knots) as compared to the IBTrACS data. Underestimation of intensity '
            'in weaker tropical cyclones can occur in ADT-HURSAT because of the necessarily '
            'relatively low resolution of HURSAT (8-km spatial and 3-hour temporal) which leads to '
            'difficulty in identifying scene type as an eye is forming or if the eye is small. '
            'ADT-HURSAT also has a lower range limit of 25 knots which is not present for IBTrACS. '
            'Note that these biases in ADT-HURSAT are consistent across the period of record in '
            'contrast to IBTrACS, where best tracking practices have varied over time.'
        )

        plt.show()

        return {'filename': 'intensity_distribution_comparison.png',
                'figure': fig,
                'alt_text': alt_text_string,
                'caption' : fig_caption}

    except Exception as e:
        print('ERROR WITH INTENSITY DISTRIBUTION')
        traceback.print_exc()
        print(e)
        return None


def generate_probability_density_plot(
    merged_data_df, basin, params):
  """
  Generates a probability density plot of 24-hour wind speed changes for
  ADT-HURSAT and IBTrACS data with external parameters.

  Constructs alt text and returns the figure object, filename,
  and alt text.

  Args:
    merged_data_df: Merged DataFrame of ADT-HURSAT and IBTrACS data (using the
                      standardized name).
    basin: The basin name for the plot title.
    params: Dictionary of plotting parameters.

  Returns:
    A dictionary containing the figure object, filename, and alt text.
  """
  alt_text_string = "Alternative text could not be generated."

  try:
    # Check if the input DataFrame is empty.
    if merged_data_df.empty:
      print(
          "No data available to generate the probability density plot for the"
          f" {basin} Basin."
      )
      alt_text_string = (
          f"Probability density plot for the {basin} Basin: No data available"
          " for the selected parameters."
      )
      print(f"Alt text: {simple_word_wrap(alt_text_string)}")
      return {
          'filename': 'probability_density_24hr_wind_change.png',
          'figure': None,
          'alt_text': alt_text_string,
      }
    # Ensure the dataframe has the necessary columns before proceeding.
    required_cols = [
        'ISO_TIME',
        'SID',
        'WindSpeed_ibtracs',
        'WindSpeed_adt_hursat',
    ]
    if not all(col in merged_data_df.columns for col in required_cols):
      print(
          "Error: Input dataframe for Probability Density Plot is missing"
          f" required columns: {', '.join(required_cols)}."
      )
      alt_text_string = (
          f"Probability density plot for the {basin} Basin: Missing required"
          " data columns."
      )
      print(f"Alt text: {simple_word_wrap(alt_text_string)}")
      return {
          'filename': 'probability_density_24hr_wind_change.png',
          'figure': None,
          'alt_text': alt_text_string,
      }

    prob_density_merged_allrecs_ibtracs_adthursat_df = merged_data_df.copy()
    # Check if 'ISO_TIME' is a valid column before setting the index.
    if 'ISO_TIME' in prob_density_merged_allrecs_ibtracs_adthursat_df.columns:
      # Ensure 'ISO_TIME' is datetime type before setting index.
      prob_density_merged_allrecs_ibtracs_adthursat_df['ISO_TIME'] = (
          pd.to_datetime(
              prob_density_merged_allrecs_ibtracs_adthursat_df['ISO_TIME'],
              errors='coerce',
          )
      )
      prob_density_merged_allrecs_ibtracs_adthursat_df.set_index(
          'ISO_TIME', inplace=True
      )

      desired_hours = params.get('hours_to_filter', [0, 6, 12, 18])
      merged_allrecs_ibtracs_adthursat_6hr_df = (
          prob_density_merged_allrecs_ibtracs_adthursat_df[
              prob_density_merged_allrecs_ibtracs_adthursat_df.index.hour.isin(
                  desired_hours
              )
          ].copy()
      )

      # Calculate intensity changes.
      merged_allrecs_ibtracs_adthursat_6hr_df[
          'Intensity_Change_Ibtracs_24hr'
      ] = merged_allrecs_ibtracs_adthursat_6hr_df.groupby('SID')[
          'WindSpeed_ibtracs'
      ].diff(
          periods=params.get('diff_periods', 4)
      )
      merged_allrecs_ibtracs_adthursat_6hr_df[
          'Intensity_Change_ADT_Hursat_24hr'
      ] = merged_allrecs_ibtracs_adthursat_6hr_df.groupby('SID')[
          'WindSpeed_adt_hursat'
      ].diff(
          periods=params.get('diff_periods', 4)
      )

      fig, ax = plt.subplots(figsize=params.get('figsize', (10, 6)))

      sns.kdeplot(
          merged_allrecs_ibtracs_adthursat_6hr_df[
              'Intensity_Change_ADT_Hursat_24hr'
          ].dropna(),
          label=params.get('adt_hursat_label', 'ADT-HURSAT'),
          fill=params.get('adt_hursat_fill', True),
          color=params.get('adt_hursat_color', '#d95f02'),
          linewidth=params.get('adt_hursat_linewidth', 2),
          ax=ax,
      )
      sns.kdeplot(
          merged_allrecs_ibtracs_adthursat_6hr_df[
              'Intensity_Change_Ibtracs_24hr'
          ].dropna(),
          label=params.get('ibtracs_label', 'IBTrACS'),
          fill=params.get('ibtracs_fill', True),
          color=params.get('ibtracs_color', '#7570b3'),
          linewidth=params.get('ibtracs_linewidth', 2),
          ax=ax,
      )

      ax.set_title(
          params.get(
              'title',
              'Probability Density 24-hour Wind Speed (Intensity) '
              f'Changes\\n {basin} Basin',
          )
      )
      ax.set_xlabel(params.get('xlabel', '24-Hour Wind Speed Change (knots)'))
      ax.set_ylabel(params.get('ylabel', 'Density'))
      ax.legend()
      ax.grid(params.get('grid', True))

      # --- Construct Alt Text ---
      try:
        #use helper function to get the acronym properly formatted for alt text
        original_title = ax.get_title()
        alt_text_title = format_acronyms_for_screen_reader(original_title)

        adt_hursat_changes = merged_allrecs_ibtracs_adthursat_6hr_df[
            'Intensity_Change_ADT_Hursat_24hr'
        ].dropna()
        ibtracs_changes = merged_allrecs_ibtracs_adthursat_6hr_df[
            'Intensity_Change_Ibtracs_24hr'
        ].dropna()

        alt_text_parts = [
            "Probability density plot titled"
            f" '{alt_text_title or 'Probability Density Plot'}'.",
            "The x-axis represents 24-Hour Wind Speed Change in knots.",
            "The y-axis represents Density.",
            (format_acronyms_for_screen_reader(
                "The plot compares the distribution of 24-hour wind speed changes"
                " for ADT-HURSAT and IBTrACS fixes.") # Updated to "fixes"
            ),
        ]

        # Add details for ADT-HURSAT distribution.
        if not adt_hursat_changes.empty:
          min_adt = adt_hursat_changes.min()
          max_adt = adt_hursat_changes.max()
          mean_adt = adt_hursat_changes.mean()
          median_adt = adt_hursat_changes.median()
          std_adt = adt_hursat_changes.std()
          adt_color = params.get('adt_hursat_color', '#d95f02')
          color_map = {
              '#d95f02': 'orange',
              '#7570b3': 'dark blue',
              # Add other color mappings as needed.
          }
          adt_color_name = color_map.get(adt_color, adt_color)

          alt_text_parts.append(
              f"{format_acronyms_for_screen_reader('ADT-HURSAT distribution is shown in')} {adt_color_name}. It ranges"
              f" from {min_adt:.1f} to {max_adt:.1f} knots, with a mean of"
              f" {mean_adt:.1f}, median of {median_adt:.1f}, and standard"
              f" deviation of {std_adt:.1f}."
          )
        else:
          alt_text_parts.append(f"{format_acronyms_for_screen_reader('ADT-HURSAT distribution: No data available.')}")

        # Add details for IBTrACS distribution.
        if not ibtracs_changes.empty:
          min_ibtracs = ibtracs_changes.min()
          max_ibtracs = ibtracs_changes.max()
          mean_ibtracs = ibtracs_changes.mean()
          median_ibtracs = ibtracs_changes.median()
          std_ibtracs = ibtracs_changes.std()
          ibtracs_color = params.get('ibtracs_color', '#7570b3')
          color_map = {
              '#d95f02': 'orange',
              '#7570b3': 'dark blue',
              # Add other color mappings as needed.
          }
          ibtracs_color_name = color_map.get(ibtracs_color, ibtracs_color)

          alt_text_parts.append(
              f"{format_acronyms_for_screen_reader('IBTrACS distribution')} is shown in {ibtracs_color_name}. It ranges"
              f" from {min_ibtracs:.1f} to {max_ibtracs:.1f} knots, with a mean of"
              f" {mean_ibtracs:.1f}, median of {median_ibtracs:.1f}, and standard"
              f" deviation of {std_ibtracs:.1f}."
          )
        else:
          alt_text_parts.append(f"{format_acronyms_for_screen_reader('IBTrACS distribution:')} No data available.")

        alt_text_string = " ".join(alt_text_parts)
        formatted_alt_text = simple_word_wrap(alt_text_string, wrap_width=80)
        print(f"Alt text: {formatted_alt_text}")

      except Exception as manual_alt_text_e:
        print(f"Error generating alt text: {manual_alt_text_e}")
        traceback.print_exc()
        alt_text_string = (
            "Alternative text could not be generated due to an error."
        )
        print(f"Alt text: {alt_text_string}")

      fig_caption = (
        'Figure Caption: This plot shows the probability density for 24-hour wind speed changes for the '
        'IBTrACS and ADT-HURSAT data sets from 1978 - 2024. Positive wind speed changes '
        'indicate that storms are intensifying by the values indicated on the x-axis in 24 hours, '
        'while negative wind speed changes indicate that storms are decreasing their wind '
        'speed by the values indicated on the x-axis. ADT-HURSAT has a narrower distribution '
        'as compared to IBTrACS, likely due to constraints on the algorithm for how quickly the '
        'intensity can increase or decrease from timestep to timestep (Dvorak rule 8 criteria).'
      )

      plt.show()

      return {
          'filename': 'probability_density_24hr_wind_change.png',
          'figure': fig,
          'alt_text': alt_text_string,
          'caption' : fig_caption
      }
    else:
      print(
          "Error: 'ISO_TIME' column not found or not in correct format after"
          " conversion in the dataframe."
      )
      alt_text_string = (
          f"Probability density plot for the {basin} Basin: Required time"
          " column is missing or in incorrect format."
      )
      print(f"Alt text: {simple_word_wrap(alt_text_string)}")
      return {
          'filename': 'probability_density_24hr_wind_change.png',
          'figure': None,
          'alt_text': alt_text_string,
      }
  except Exception as e:
    print('ERROR WITH PROBABILITY DENSITY')
    traceback.print_exc()
    alt_text_string = "Alternative text could not be generated due to an error."
    print(f"Alt text: {simple_word_wrap(alt_text_string)}")
    return {
        'filename': 'probability_density_24hr_wind_change.png',
        'figure': None,
        'alt_text': alt_text_string,
    }



def generate_distribution_TC_max_intensity_plot(merged_data_df, basin, params):
    """
    Maximum tropical cyclone intensity per storm comparing the distribution
    for ADT-HURSAT and IBTrACS data with external parameters and returns
    the figure object and filename.
    Bins are calculated based on a defined 'bin_interval' from the parameters.

    Args:
        merged_data_df: Merged DataFrame of ADT-HURSAT and IBTrACS data
           (using the standardized name).
        basin: The basin name for the plot title.
        params: Dictionary of plotting parameters.

    Returns:
        A dictionary containing the figure object and filename.
    """
    alt_text_string = "Alternative text could not be generated."

    try:
        # Check if the input DataFrame is empty
        if merged_data_df.empty:
            print(f"No data available to generate the distribution of maximum tropical"
                  f"cyclone intensity plot for the {basin} Basin.")
            alt_text_string = (
                f"Histogram plot for the {basin} Basin: No data available for the selected parameters."
            )
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {'filename': 'max_intensity_distribution_by_sid.png', 'figure': None, 'alt_text': alt_text_string}

        # Ensure the dataframe has the necessary columns before proceeding
        required_cols = ['SID', 'WindSpeed_adt_hursat', 'WindSpeed_ibtracs']
        if not all(col in merged_data_df.columns for col in required_cols):
            print(f"Error: Input dataframe for Max Intensity Distribution Plot is missing "
                  f"required columns: {', '.join(required_cols)}.")
            alt_text_string = (
                f"Histogram plot for the {basin} Basin: Missing required data columns."
            )
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")
            return {'filename': 'max_intensity_distribution_by_sid.png', 'figure': None, 'alt_text': alt_text_string}

        print(f"Number of SID's = {len(merged_data_df['SID'].unique())}")

        adt_hursat_intensity_by_sid = merged_data_df.groupby(
           merged_data_df['SID'])['WindSpeed_adt_hursat'].max()
        ibtracs_intensity_by_sid = merged_data_df.groupby(
           merged_data_df['SID'])['WindSpeed_ibtracs'].max()

        adt_hursat_data = adt_hursat_intensity_by_sid.dropna()
        ibtracs_data = ibtracs_intensity_by_sid.dropna()

        bin_interval = params.get('bin_interval', 5)

        # Combine data to find the overall min/max
        combined_data = pd.concat([adt_hursat_data, ibtracs_data])
        # Find minimum and maximum values, rounding down/up to the nearest interval
        data_min = np.floor(combined_data.min() / bin_interval) * bin_interval
        data_max = np.ceil(combined_data.max() / bin_interval) * bin_interval

        # Use numpy.arange to create fixed-interval bins
        bins_to_use = np.arange(data_min, data_max + bin_interval, bin_interval)
        # --- END NEW BIN CALCULATION LOGIC ---

        fig, ax = plt.subplots(figsize=params.get('figsize', (10, 6)))

        # Initialize variables to hold histogram counts
        adt_counts = []
        ibtracs_counts = []

        if not adt_hursat_data.empty:
            # Capture the counts and bins from the histogram plot
            adt_counts, adt_bins, _ = ax.hist(adt_hursat_data,
                    bins=bins_to_use, # <- REFRACTORED LINE
                    alpha=params.get('adt_hursat_alpha', 0.7),
                    label=params.get('adt_hursat_label', 'ADT-HURSAT'),
                    color=params.get('adt_hursat_color', '#d95f02'))
        else:
            print("No non-NaN values in ADT-HURSAT max intensity data for the selected "
                  f"basin. Skipping ADT-HURSAT histogram.")

        # Check if there are any non-NaN values in the ibtracs_intensity_by_sid before plotting
        if not ibtracs_data.empty:
            # Capture the counts and bins from the histogram plot
            ibtracs_counts, ibtracs_bins, _ = ax.hist(ibtracs_data,
                    bins=bins_to_use, # <- REFRACTORED LINE
                    alpha=params.get('ibtracs_alpha', 0.7),
                    label=params.get('ibtracs_label', 'IBTrACS'),
                    color=params.get('ibtracs_color', '#7570b3'))
        else:
            print("No non-NaN values in IBTrACS max intensity data for the selected basin. "
                  f"Skipping IBTrACS histogram.")

        ax.set_title(params.get('title', f'Distribution of Tropical Cyclone Intensity, '
                                f'by Storm ID (SID) for entire period, {basin} Basin'))
        ax.set_xlabel(params.get('xlabel', 'Intensity (knots)'))
        ax.set_ylabel(params.get('ylabel', 'Frequency'))
        ax.legend()
        ax.grid(axis='y', linestyle='--', alpha=0.6) # Added grid for clarity

        # NEW: Customize x-axis ticks based on bin_interval
        if bin_interval > 0:
             # If bin_interval is small (e.g., 5), show every 2nd interval (10)
             # If bin_interval is larger (e.g., 10), show every 2nd interval (20)
             tick_interval = bin_interval * 2 if bin_interval < 20 else bin_interval
             ax.xaxis.set_major_locator(ticker.MultipleLocator(tick_interval))

        # --- Construct Alt Text ---
        try:
            #use helper function to get the acronym properly formatted for alt text
            original_title = ax.get_title()
            alt_text_title = format_acronyms_for_screen_reader(original_title)

            # Map common hex colors to names for better readability in alt text
            color_map = {
                '#d95f02': 'orange',
                '#7570b3': 'dark blue',
                '#0076D6': 'blue',
                '#707070': 'gray',
            }

            alt_text_parts = [
                f"Histogram plot titled '{alt_text_title}'.",
                f"The x-axis represents {ax.get_xlabel()} from approximately"
                f" {data_min:.0f} to {data_max:.0f}, with a bin interval of {bin_interval} knots."
            ]

            # Describe the Y-axis range
            y_axis_description = f"The y-axis represents {ax.get_ylabel()}."
            if len(adt_counts) > 0 or len(ibtracs_counts) > 0:
                overall_max_freq = max(adt_counts.max() if len(adt_counts) > 0 else 0,
                                       ibtracs_counts.max() if len(ibtracs_counts) > 0 else 0)
                y_axis_description += (f" The frequency, or count of observations per bin,"
                                       f" ranges up to approximately {int(overall_max_freq)}.")
            alt_text_parts.append(y_axis_description)

            # Describe the ADT-HURSAT data if it exists
            if not adt_hursat_data.empty:
                adt_color_hex = params.get('adt_hursat_color', '#d95f02')
                adt_color_name = color_map.get(adt_color_hex, adt_color_hex)
                alt_text_parts.append(
                    f"{format_acronyms_for_screen_reader('One dataset, labeled ADT-HURSAT')} and colored {adt_color_name},"
                    f" shows a distribution with an average intensity of {adt_hursat_data.mean():.1f} knots"
                    f" and a peak frequency of {int(adt_counts.max())} observations in a single intensity bin."
                )

            # Describe the IBTrACS data if it exists
            if not ibtracs_data.empty:
                ibtracs_color_hex = params.get('ibtracs_color', '#7570b3')
                ibtracs_color_name = color_map.get(ibtracs_color_hex, ibtracs_color_hex)
                alt_text_parts.append(
                    f"{format_acronyms_for_screen_reader('A second dataset, labeled IBTrACS')} and colored {ibtracs_color_name},"
                    f" shows a distribution with an average intensity of {ibtracs_data.mean():.1f} knots"
                    f" and a peak frequency of {int(ibtracs_counts.max())} observations in a single intensity bin."
                )

            alt_text_string = " ".join(alt_text_parts)
            formatted_alt_text = simple_word_wrap(alt_text_string, wrap_width=80)
            print(f"Alt text: {formatted_alt_text}")

        except Exception as alt_text_e:
            print(f"Error generating alt text: {alt_text_e}")
            traceback.print_exc()
            alt_text_string = "Alternative text could not be generated due to an error."
            print(f"Alt text: {simple_word_wrap(alt_text_string)}")

        fig_caption = (
            'Figure Caption: This plot shows a comparison of the distribution of tropical cyclone lifetime '
            'maximum intensity, between the ADT-HURSAT and IBTrACS data for the period '
            '1978-2024 in the North Atlantic Basin. The lifetime maximum intensity is found by '
            'grouping the tropical cyclones by Storm ID (SID) then finding the highest value of the '
            'maximum sustained wind speeds per storm. Similar to the distributions for all fixes (5.1), '
            'the ADT-HURSAT data has fewer minor hurricanes (65-96 knots) and more tropical '
            'storms (<64 knots) as compared to IBTrACS, since ADT-HURSAT sometimes has a '
            'hard time identifying eye and pinhole eye scenes due to the 8-km resolution of HURSAT.'

        )

        plt.show()

        return {'filename': 'max_intensity_distribution_by_sid.png',
                'figure': fig,
                'alt_text': alt_text_string,
                'caption' : fig_caption}

    except Exception as e:
        print('ERROR WITH DISTRIBUTION OF MAX INTENSITY')
        traceback.print_exc()
        print(e)
        return None


## --3-- Data Ingestion and Transformation Pipeline

This section executes the data processing pipeline using the utility functions defined in Section 2. The workflow proceeds in logical stages:

 - Load Reference Data: Ingesting the IBTrACS "best track" dataset and auxiliary files (storm names, boundary images).

 - Query Cloud Storage: Retrieving the full inventory of ADT-HURSAT files from the Google Cloud Bucket.

 - Filter and Subset: narrowing the file list to a specific basin (e.g., North Atlantic) to manage processing time and scope.

 - Parallel Loading: Reading thousands of NetCDF files concurrently into a structured DataFrame.

 - Data Transformation: Cleaning, rounding timestamps, and merging datasets to prepare for visualization.

### --3.1-- Load IBTrACS Reference Data
First, we load the International Best Track Archive for Climate Stewardship (IBTrACS) dataset. This serves as the ground truth for our comparisons. We read this directly from a cloud-hosted NetCDF file.

**Note:** This notebook uses IBTrACS data that best matches ADT-HURSAT for consistent scientific comparison, but it is **not** the most up-to-date live data. If you wish to perform separate analyses using the latest available data, please refer to the official [IBTrACS website](https://www.ncei.noaa.gov/products/international-best-track-archive).

In [ ]:
'''
3.1. Read in the IBTrACS data
'''

#this is the public url for the data file
ibtracs_url = 'https://storage.googleapis.com/noaa-ncei-ipg/notebooks/data/' \
    'hursat/adt/IBTrACS.forhursat7b.v04r01.nc'
ibtracs_dataframe = read_ibtracs_data_from_url(ibtracs_url)

group_basin_counts = ibtracs_dataframe.groupby('BASIN').size()
print(f"Total number of IBTrACS records: {len(ibtracs_dataframe)}")
print(f"Number of IBTrACS records by Basin:")
print(group_basin_counts)

### --3.2-- Read the supporting IBTrACS names file and the basin boundary image
This step loads two critical helper files:

 - Storm ID Mapping: A CSV file that maps storm identifiers between the ADT-HURSAT and IBTrACS datasets, ensuring accurate merging.

 - Basin Boundary Image: A specialized image array used by our get_basin() utility to resolve geographical ambiguities between the North Atlantic and Eastern Pacific basins.

In [ ]:
'''
3.2 Read the supporting IBTrACS names file and the basin boundary image
'''

# URL of the CSV file with supporting IBTrACS names and IDs
adt_hursat_storms_csv_file = "https://storage.googleapis.com/noaa-ncei-ipg/" \
    "notebooks/data/hursat/adt/ADT-HURSAT_list_of_storm_id.csv"

ibtracs_names = read_ibtracs_names_file(adt_hursat_storms_csv_file)

# URL of the boundary image file
basin_image_url = "https://storage.googleapis.com/noaa-ncei-ipg/notebooks/" \
    "data/hursat/adt/Basin_EPacs_Natl.png"

epac_natl_img_array = read_boundary_image(basin_image_url)

### --3.3-- Access ADT-HURSAT data bucket to get a list of data
Here we connect to the Google Cloud Storage bucket to retrieve the complete list of available ADT-HURSAT file paths.

Note: This operation lists all available files (often >4,000). In the next step, we will filter this list to a manageable subset.

In [ ]:
'''
3.3 Access ADT-HURSAT data bucket to get a listing of data blobs within
Here the user will pull a listing of ADT HURSAT files from the Google Cloud Bucket.
'''

adt_hursat_files_path_list = get_list_adt_hursat_files()

print(f"Number of ADT-HURSAT files: {len(adt_hursat_files_path_list)}")
print("This is the entire listing of files in the bucket!\nYou need to either "
        "subset this list by basin below or be\nprepared to wait up to 30 "
        "minutes in Section 3.5 to load the entire\ndataset into a dataframe.")


### --3.4-- Filter ADT-HURSAT and IBTrACS by Basin
To keep processing times reasonable and focus the analysis, we filter the master file list to a specific basin.

Default: The code below filters for the North Atlantic (NA) basin.

Customization: You can change basin_to_filter to other codes (e.g., 'EP', 'WP') to analyze different regions.

In [ ]:
'''
3.4 Filter ADT-HUSAT file list by Basin and filter the IBTrACS dataframe by Basin

Basin NAMES
 NA = North Atlantic
 SA = South Atlantic
 NI = North Indian
 SI = South Indian
 WP = Western North Pacific
 EP = Eastern North Pacific
 SP = South Pacific
'''
basin_to_filter = 'NA'
basin_name_for_plots = get_basin_name(basin_to_filter)

adt_hursat_basin_filtered_list = filter_data_list_by_basin(
    adt_hursat_files_path_list, basin_to_filter, epac_natl_img_array)

basin_filtered_ibtracs_dataframe = filter_ibtracs_dataframe_by_basin(
    ibtracs_dataframe, basin_to_filter)

#print a check on the basin and storm counts from both products after the basin filtering
indv_basin_counts = basin_filtered_ibtracs_dataframe.groupby('BASIN').size()
print(indv_basin_counts)
print(basin_filtered_ibtracs_dataframe['BASIN'].unique())
print(basin_name_for_plots)
print(f"Number of ADT-HURSAT files filtered by {basin_to_filter} basin:" \
      f"{len(adt_hursat_basin_filtered_list)}")
print(f'Number of ADT-HURSAT files in the original full list: {len(adt_hursat_files_path_list)}')


### --3.5-- Load ADT-HURSAT data into a dataframe
This is the most computationally intensive step. We use the build_adt_hursat_dataframe_parallel utility to open, read, and aggregate hundreds of NetCDF files concurrently.

Performance: Loading the North Atlantic basin (~700+ files) typically takes 2-5 minutes depending on network speed.

In [ ]:
%%time
'''
3.5 Load ADT-HURSAT data into a datframe
'''

print(f"""***Dataframe is being built using data from the {basin_to_filter} basin.***
This could take several minutes.
The number of files in the basin filtered list: {len(adt_hursat_basin_filtered_list)} \n""")

adt_hursat_dataframe = build_adt_hursat_dataframe_parallel(adt_hursat_basin_filtered_list)

### --3.6-- Transform ADT-HURSAT basin dataframe for plots
Once the raw data is loaded, we apply specific transformations to prepare it for plotting:

 - Hurricane Fixes: Extracting data at 6 hourly intervals to match initial source agency best track data

 - Intensity Calculations: creating boolean flags for Major Hurricane (>96 kts) and Hurricane (>64 kts) thresholds.

 - Aggregation: Generating annual counts for the bar charts.

In [ ]:
'''
3.6 Subset and transform original ADT-HURSAT data frame for ADT-HURSAT plotting
These data frames are used in the subsequent plots.
***
If an error occurs when plotting or output is not as expected, first try to run this cell
again to verify the dataframes required are loaded
***
'''
print(f"***Dataframe is being built using data from the {basin_to_filter} basin.*** \n" \
      f"The number of files in the basin filtered list going into the " \
        f"transformations: {len(adt_hursat_basin_filtered_list)} \n""")

adt_hursat_counts_transformed_dataframes = transform_adt_hursat_dataframe(
    adt_hursat_dataframe,ibtracs_names,basin_to_filter)
adt_hursat_counts_df_for_barchart = adt_hursat_counts_transformed_dataframes[0]
adt_hursat_plots_compiled_df = adt_hursat_counts_transformed_dataframes[1]
adt_hursat_ibtracs_compiled_df = adt_hursat_plots_compiled_df.copy()

### --3.7-- Transform IBTrACS dataframe and integrate with ADT-HURSAT dataframe for comparison plots
Finally, we merge the processed ADT-HURSAT data with the IBTrACS reference data. This creates a unified DataFrame (merged_ibtracs_adthursat_dataframe) containing coincident records, essential for the direct comparison plots in Section 5.

In [ ]:
'''
3.7 Build dataframe for comparison plots
'''
print(f"""***Dataframe is being built combining data from ADT-HURSAT and IBTrACS data sets, and
using data from the {basin_to_filter} basin.***""")

merged_ibtracs_adthursat_dataframe = build_adt_hursat_ibtracs_graphs_dataframe(
    adt_hursat_ibtracs_compiled_df, basin_filtered_ibtracs_dataframe)


## --4-- ADT-HURSAT data plots
This section generates visualizations derived exclusively from the ADT-HURSAT dataset. These plots characterize the dataset's internal climatology and trends.

Accessibility Note: All plotting functions called here automatically generate detailed Alt Text to ensure compliance with accessibility standards.

Saving Plots: Code blocks include commented-out instructions to save figures locally. Uncomment these lines if you wish to export the images.

<i>Each data plot has code at the bottom of the cell to save the plot. Those plots <br>
are saved within the Colab runtime environment folder called 'content', and by selecting <br>
the 'Files' icon (folder symbol) at the far left of the notebook, you can download each <br>
file to your local machine. The three vertical dots at the end of each file <br> 
(when hovering over the file name in the contents folder) or a left mouse click on the <br>
file will give you the options to work with the saved plot files. <br>

### --4.1-- Generate a bar chart from ADT-HURSAT filtered data
This cell generates a grouped bar chart displaying the annual frequency of storms across three intensity categories: Named Storm Fixes, Hurricane Fixes, and Major Hurricane Fixes. This provides a high-level view of storm activity over the decades.

In [ ]:
'''
4.1 Generate a bar chart from ADT-HURSAT filtered data
'''

# Define parameters for the Bar Chart
bar_chart_adt_hursat_storm_counts_params = {
    'figsize': (12, 6),
    'color_map': {
        'Named_Storms': '#1b9e77',
        'Hurricanes': '#d95f02',
        'Major_Hurricanes': '#7570b3'
    },
    'bar_width': 0.2,
    'group_spacing': 0.01,
    'title': (
        f'ADT-HURSAT Storm Counts, 1978 - 2024 Climatology,'
        f' {basin_name_for_plots} Basin'
    ),
    'xlabel': 'Year',
    'ylabel': 'Annual Count',
    'xticks_rotation': 45,
    'xticks_ha': 'right',
    'legend_title': 'Category',
    'major_tick_spacing': 10,  # Set major tick spacing to 10
    'minor_tick_spacing': 2,   # Set minor tick spacing to 2
    'minor_yticks_fontsize': 8 # Added font size for minor y-ticks
}

bar_chart_adt_hursat_storm_counts_plot_dict = (
    generate_bar_chart_adt_hursat_storm_counts(
        adt_hursat_counts_df_for_barchart, basin_name_for_plots,
        bar_chart_adt_hursat_storm_counts_params))
print(textwrap.fill(bar_chart_adt_hursat_storm_counts_plot_dict["caption"], width=80))
# If you'd like a custom file name change the file_name variable: file_name = 'yourcustomfilename.png
# If you would like to save out the figure uncomment the following lines
# if bar_chart_adt_hursat_storm_counts_plot_dict and bar_chart_adt_hursat_storm_counts_plot_dict['figure']:
#   file_name = bar_chart_adt_hursat_storm_counts_plot_dict['filename']
#   bar_chart_adt_hursat_storm_counts_plot_dict['figure'].savefig(file_name, bbox_inches='tight')

### --4.2-- Generate proportional intensities time series from ADT-HURSAT filtered data
This analysis plots the annual proportion of Major Hurricanes relative to all Hurricanes.

 - Methodology: Data is binned by year. For each annual bin, we count the number of Major Hurricane fixes and divide it by the total count of Hurricane fixes: (Count of Major Fixes) / (Total Count of Hurricane Fixes).

 - Goal: To visualize if the relative frequency of major storms is changing over time, independent of the total number of storms.

In [ ]:
'''
4.2 Generate proportional intensities time series from ADT-HURSAT filtered data
'''

# Define parameters for the Proportional Intensities Time Series Plot
proportional_intensities_adt_hursat_time_series_params = {
    'figsize': (14, 6),
    'line_color': '#0076D6',
    'marker_color': '#0076D6',
    'trend_line_color': '#707070',
    'trend_line_linestyle': '--',
    'xlabel': 'Year',
    'ylabel': 'Proportion of major hurricane fixes to all hurricane fixes',
    'title': (
        'ADT-HURSAT Proportion of major hurricane fixes to all hurricane fixes,'
        f' {basin_name_for_plots} Basin'
    ),
    'legend_loc': 'lower center',
    'legend_bbox_to_anchor': (0.5, -0.3),
    'legend_ncol': 6,
    'legend_fontsize': 7,
    'grid': True,
    'xticks_rotation': 45,
    'xticks_ha': 'right',
    'xticks_fontsize': 7,
    'resample_period': '1YE'
}


proportional_intensities_adt_hursat_time_series_plot_dict = (
    generate_proportional_intensities_adt_hursat_time_series(
        adt_hursat_plots_compiled_df, basin_name_for_plots,
        proportional_intensities_adt_hursat_time_series_params))
print(textwrap.fill(proportional_intensities_adt_hursat_time_series_plot_dict["caption"], width=80))

# If you'd like a custom file name change the file_name variable: file_name = 'yourcustomfilename.png
# If you would like to save out the figure uncomment the following lines
# if (proportional_intensities_adt_hursat_time_series_plot_dict and
#     proportional_intensities_adt_hursat_time_series_plot_dict['figure']):
#   file_name = proportional_intensities_adt_hursat_time_series_plot_dict['filename']
#   proportional_intensities_adt_hursat_time_series_plot_dict['figure'].savefig(file_name, bbox_inches='tight')

### --4.3-- Generate wind speed time series from ADT-HURSAT filtered data
This plot displays the annual mean wind speed for all storm fixes meeting the Named Storm threshold (>= 34 kts).

 - Methodology: Data is binned by year. For each annual bin, we calculate the arithmetic mean (average) of the wind speeds for all observations in that year.

 - Goal: To identify variation or trends in the intensity of all storm fixes over time.

In [ ]:
'''
4.3 Generate a wind speed time series plot from ADT-HURSAT filtered data
'''

# Define parameters for the Wind Speed Time Series Plot
wind_speed_time_series_params = {
    'figsize': (14, 6),
    'line_color': '#0076D6',
    'trend_line_color': '#707070',
    'trend_line_linestyle': '--',
    'xlabel': 'Year',
    'ylabel': 'WindSpeed (knots)',
    'title': (f'ADT-HURSAT Wind Speed (knots) Time Series for all fixes >= 34 kts, '
             f'{basin_name_for_plots} Basin'),
    'legend_loc': 'lower center',
    'legend_bbox_to_anchor': (0.5, -0.3),
    'legend_ncol': 6,
    'legend_fontsize': 7,
    'grid': True,
    'xticks_rotation': 45,
    'xticks_ha': 'right',
    'xticks_fontsize': 7,
    'resample_period': '1YE'  # Resampling period (e.g., '1YE' for 1-year end)
}

wind_speed_time_series_plot_dict = (
    generate_wind_speed_time_series(
        adt_hursat_plots_compiled_df, basin_name_for_plots,
        wind_speed_time_series_params))
print(textwrap.fill(wind_speed_time_series_plot_dict["caption"], width=80))

# If you'd like a custom file name change the file_name variable: file_name = 'yourcustomfilename.png
# If you would like to save out the figure uncomment the following lines
# if wind_speed_time_series_plot_dict and wind_speed_time_series_plot_dict['figure']:
#   file_name = wind_speed_time_series_plot_dict['filename']
#   wind_speed_time_series_plot_dict['figure'].savefig(file_name, bbox_inches='tight')

### --4.4-- Generate time series all hurricane fixes (>=64 kts)
Filtering strictly for Hurricane-force fixes (>= 64 kts), this time series visualizes the annual mean wind speed of stronger storms.

 - Methodology: We first filter the dataset to include only hurricane-strength fixes. The remaining data is then binned by year, and we calculate the mean wind speed for each year.

 - Goal: To identify variation or trends in the intensity of all hurricane fixes over time.

In [ ]:
'''
4.4 Generate time series plot for all hurricane fixes (>=64 kts).
'''

# Define parameters for the All Hurricane Fixes Time Series Plot
all_hurricanes_time_series_params = {
    'figsize': (14, 6),
    'line_color': '#0076D6',
    'trend_line_color': '#707070',
    'trend_line_linestyle': '--',
    'xlabel': 'Year',
    'ylabel': 'WindSpeed (knots)',
    'title': (
        'ADT-HURSAT Wind Speed (knots) Time Series for all hurricane fixes,'
        f' {basin_name_for_plots} Basin'
    ),
    'legend_loc': 'lower center',
    'legend_bbox_to_anchor': (0.5, -0.3),
    'legend_ncol': 6,
    'legend_fontsize': 7,
    'grid': True,
    'xticks_rotation': 45,
    'xticks_ha': 'right',
    'xticks_fontsize': 7,
    'resample_period': '1YE',  # Resampling period (e.g., '3YE' for 3-year end)
}

all_hurricanes_time_series_plot_dict = (
    generate_time_series_all_hurricanes(
        adt_hursat_plots_compiled_df,
        basin_name_for_plots,
        all_hurricanes_time_series_params,
    )
)
print(textwrap.fill(all_hurricanes_time_series_plot_dict["caption"], width=80))

# If you'd like a custom file name change the file_name variable: file_name = 'yourcustomfilename.png
# If you would like to save out the figure uncomment the following lines
# if all_hurricanes_time_series_plot_dict and all_hurricanes_time_series_plot_dict['figure']:
#   file_name = all_hurricanes_time_series_plot_dict['filename']
#   all_hurricanes_time_series_plot_dict['figure'].savefig(
#       file_name, bbox_inches='tight'
#   )

### --4.5-- Generate time series of major hurricane fixes (>=96 kts)
Filtering strictly for Major Hurricane fixes (>= 96 kts), this plot tracks the annual mean wind speed of the most intense systems.

 - Methodology: We first filter the dataset for major hurricane fixes. The data is then binned by year, and we calculate the mean wind speed for each year.

 - Goal: To identify variation or trends in the intensity of major hurricane fixes over time.

In [ ]:
'''
4.5 Generate time series plot for major hurricane fixes (>=96 kts).
'''

# Define parameters for the Major Hurricanes Time Series Plot
major_hurricanes_time_series_params = {
    'figsize': (14, 6),
    'line_color': '#0076D6',
    'trend_line_color': '#707070',
    'trend_line_linestyle': '--',
    'xlabel': 'Year',
    'ylabel': 'WindSpeed (knots)',
    'title': f'ADT-HURSAT Wind Speed (knots) Time Series for all major hurricane fixes, {basin_name_for_plots} Basin',
    'legend_loc': 'lower center',
    'legend_bbox_to_anchor': (0.5, -0.3),
    'legend_ncol': 6,
    'legend_fontsize': 7,
    'grid': True,
    'xticks_rotation': 45,
    'xticks_ha': 'right',
    'xticks_fontsize': 7,
    'resample_period': '1YE', # Resampling period (e.g., '3YE' for 3-year end)
}

major_hurricanes_time_series_plot_dict = generate_time_series_major_hurricanes(
    adt_hursat_plots_compiled_df, basin_name_for_plots, major_hurricanes_time_series_params)
print(textwrap.fill(major_hurricanes_time_series_plot_dict["caption"], width=80))

# If you'd like a custom file name change the file_name variable: file_name = 'yourcustomfilename.png
# If you would like to save out the figure uncomment the following lines
# if major_hurricanes_time_series_plot_dict and major_hurricanes_time_series_plot_dict['figure']:
#   file_name = major_hurricanes_time_series_plot_dict['filename']
#   major_hurricanes_time_series_plot_dict['figure'].savefig(file_name, bbox_inches='tight')

### --4.6-- Generate density normalization histogram
This cell generates a density-normalized histogram of all wind speed observations. It reveals the overall distribution of storm intensities, helping to identify the most common intensity ranges (modes) in the dataset.

In [ ]:
'''
4.6 Generate a density normalized histogram plot from ADT-HURSAT filtered data
'''

# Define parameters for the Density Normalization Histogram
density_norm_histo_params = {
    'figsize': (10, 6),
    'bins': 20,
    'density': True,
    'color': '#084fc9',
    'alpha': 0.5,
    'title': f'ADT-HURSAT Histogram of Wind Speed (Density Normalized), {basin_name_for_plots} Basin',
    'xlabel': 'Wind Speed (knots)',
    'ylabel': 'Density',
    'grid': True
    }

density_norm_histo_plot_dict = generate_density_norm_histo(
    adt_hursat_plots_compiled_df, basin_name_for_plots, density_norm_histo_params)
print(textwrap.fill(density_norm_histo_plot_dict["caption"], width=80))

# If you'd like a custom file name change the file_name variable: file_name = 'yourcustomfilename.png
# If you would like to save out the figure uncomment the following lines
# if density_norm_histo_plot_dict and density_norm_histo_plot_dict['figure']:
#   file_name = density_norm_histo_plot_dict['filename']
#   density_norm_histo_plot_dict['figure'].savefig(file_name, bbox_inches='tight')

## --5-- IBTrACS and ADT-HURSAT comparison plots
This section validates the ADT-HURSAT dataset by comparing it directly against the IBTrACS "best track" standard. These visualizations highlight agreement, bias, and distributional differences between the satellite-derived data (ADT) and the comprehensive, global collection of tropical cyclones (IBTrACS).

<i>Each data plot has code at the bottom of the cell to save the plot. Those plots <br>
are saved within the Colab runtime environment folder called 'content', and by selecting <br>
the 'Files' icon (folder symbol) at the far left of the notebook, you can download each <br>
file to your local machine. The three vertical dots at the end of each file <br> 
(when hovering over the file name in the contents folder) or a left mouse click on the <br>
file will give you the options to work with the saved plot files. <br>

### --5.1-- Generate comparison distribution plot of tropical cyclone intensity for ADT-HURSAT and IBTrACS
This histogram overlays the wind speed distributions of both datasets.

In [ ]:
'''
5.1 Generate a histogram comparing the distribution of tropical cyclone intensity
for the ADT-HURSAT and IBTrACS data
'''

# Define parameters for the Intensity Distribution Comparison Plot
intensity_distribution_params = {
    'figsize': (10, 6),
    'bin_interval': 5,
    'adt_hursat_alpha': 0.7,
    'adt_hursat_label': 'ADT-HURSAT',
    'adt_hursat_color': '#d95f02',
    'ibtracs_alpha': 0.7,
    'ibtracs_label': 'IBTrACS',
    'ibtracs_color': '#7570b3',
    'title': f'Distribution of Tropical Storm Intensity for entire period, {basin_name_for_plots} Basin',
    'xlabel': 'Intensity (knots)',
    'ylabel': 'Frequency'
}

intensity_distribution_plot_dict = generate_intensity_distribution_plot(
    merged_ibtracs_adthursat_dataframe, basin_name_for_plots, intensity_distribution_params)
print(textwrap.fill(intensity_distribution_plot_dict["caption"], width=80))


# If you'd like a custom file name change the file_name variable: file_name = 'yourcustomfilename.png
# If you would like to save out the figure uncomment the following lines
# if intensity_distribution_plot_dict and intensity_distribution_plot_dict['figure']:
#   file_name = intensity_distribution_plot_dict['filename']
#   intensity_distribution_plot_dict['figure'].savefig(file_name, bbox_inches='tight')

### --5.2-- Generate comparison probability density plot of 24 hour wind speed changes for ADT-HURSAT and IBTrACS
This plot compares how well each dataset captures rapid intensity changes (intensification or weakening) over a 24-hour window.

In [ ]:
'''
5.2 Generate probability density plot of 24-hour wind speed (intensity) changes.
'''

# Define parameters for the Probability Density Plot (Alt Text Version)
probability_density_params = {
    'figsize': (10, 6),
    'hours_to_filter': [0, 6, 12, 18],
    'diff_periods': 4,
    'adt_hursat_label': 'ADT-HURSAT',
    'adt_hursat_fill': True,
    'adt_hursat_color': '#d95f02',
    'adt_hursat_linewidth': 2,
    'ibtracs_label': 'IBTrACS',
    'ibtracs_fill': True,
    'ibtracs_color': '#7570b3',
    'ibtracs_linewidth': 2,
    'title': (
        'Probability Density 24-hour Wind Speed (Intensity) Changes\n'
        f' {basin_name_for_plots} Basin'
    ),
    'xlabel': '24-Hour Wind Speed Change (knots)',
    'ylabel': 'Density',
    'grid': True,
}

probability_density_plot_dict = (
    generate_probability_density_plot(
        merged_ibtracs_adthursat_dataframe,
        basin_name_for_plots,
        probability_density_params,
    )
)
print(textwrap.fill(probability_density_plot_dict["caption"], width=80))


# If you'd like a custom file name change the file_name variable: file_name = 'yourcustomfilename.png
# If you would like to save out the figure uncomment the following lines
# if (
#     probability_density_plot_dict
#     and probability_density_plot_dict['figure']
#     ):
#   file_name = probability_density_plot_dict['filename']
#   probability_density_plot_dict['figure'].savefig(
#       file_name, bbox_inches='tight'
#       )

### --5.3-- Generate distribution plot of maximum tropical cyclone intensity for ADT-HURSAT and IBTrACS
This histogram compares the Lifetime Maximum Intensity recorded for each unique storm ID (SID) in both datasets.

In [ ]:
'''
5.3 Generate a histogram comparing the distribution of maximum tropical cyclone intensity
for the ADT-HURSAT and IBTrACS data
'''

# Define parameters for the Distribution of Maximum Intensity Plot
max_intensity_distribution_params = {
    'figsize': (10, 6),
    'bin_interval': 5,
    'adt_hursat_alpha': 0.7,
    'adt_hursat_label': 'ADT-HURSAT',
    'adt_hursat_color': '#d95f02',
    'ibtracs_alpha': 0.7,
    'ibtracs_label': 'IBTrACS',
    'ibtracs_color': '#7570b3',
    'title': f'Distribution of Tropical Cyclone Lifetime Maximum Intensity for ' \
         f' the entire period\n {basin_name_for_plots} Basin',
    'xlabel': 'Intensity (knots)',
    'ylabel': 'Frequency'
}

max_intensity_distribution_plot_dict = generate_distribution_TC_max_intensity_plot(
    merged_ibtracs_adthursat_dataframe, basin_name_for_plots, max_intensity_distribution_params)
print(textwrap.fill(max_intensity_distribution_plot_dict["caption"], width=80))

# If you'd like a custom file name change the file_name variable: file_name = 'yourcustomfilename.png
# If you would like to save out the figure uncomment the following lines
# if max_intensity_distribution_plot_dict:
#   file_name = max_intensity_distribution_plot_dict['filename']
#   max_intensity_distribution_plot_dict['figure'].savefig(file_name, bbox_inches='tight')

## --6-- Filter ADT-HURSAT Data Files  
These utilities allow for granular file exploration. While the main analysis (Sections 3-5) relies on basin-wide data, these cells help you identify and list specific files based on narrower criteria (Year, Month, or Basin).

Usage: These filters return lists of file URLs. You can use these lists to inspect specific events or download small subsets of data for offline analysis.

<i>If you filter below by year, month or basin-year-month, THE BASIN-WIDE,<br> 
plots and time series in Sections 4 and 5 will not work as expected. These filters are best for <br>
defining files for downloading data if desired. Depending on the number of files <br> 
returned from the filtering, there may be problems encountered with the plots <br>
due to lack of an adequate range of values to plot.<br>

In the following code blocks, you will work through the process to filter the <br>
ADT-HURSAT data files in the Google Cloud Bucket by defining various filters. <br>

### --6.1-- Filter All Storms by Basin
Filters the master list of files to return only those belonging to a specific basin (e.g., 'NI' for North Indian). Useful for checking data availability in other regions without running the full pipeline.

<i>Here you can filter the entire listing of ADT-HURSAT files by Basin. <br>
The resulting list named 'adt_hursat_datalist_filtered_by_basin' can then be used<br>
To further filter by year and month later in cell 6.4

In [ ]:
'''
6.1 Filter All Storms by Basin

Basin NAMES
 NA = North Atlantic
 SA = South Atlantic
 NI = North Indian
 SI = South Indian
 WP = Western North Pacific
 EP = Eastern North Pacific
 SP = South Pacific
'''
basin_filter_for_datalist = 'NI'

adt_hursat_datalist_filtered_by_basin = filter_data_list_by_basin(
    adt_hursat_files_path_list, basin_filter_for_datalist, epac_natl_img_array)

print(f"Files filtered by basin {basin_filter_for_datalist}:")
display(adt_hursat_datalist_filtered_by_basin[:10])

### --6.2-- Filter All Storms by Year
Filters the file list to return only storms occurring in a specific calendar year (e.g., 2023). Useful for annual case studies.

<i>Here you will filter the storms by year and store these filenames in a list. <br>
The filename contains the year in which the storm occurred. <br>
This information can be used to filter for all storms that occurred in a single year.<br>

This example uses 2023 as the year of interest. You can change this to any year from 1978 through 2024.

In [ ]:
'''
6.2 Filter All Storms by Year
This example uses 2022 as the year of interest. You can change this to any year from 1978 through 2024.
After running this cell, use the 'year_filtered_list' below to run cell 2.5 to load these data files into dataframe
'''
yyyy = '2023'

adt_hursat_datalist_filtered_by_year = filter_files_by_year(adt_hursat_files_path_list, yyyy)

# Display the first few elements of the filtered list
print(f"Files filtered by year {yyyy}:")
display(adt_hursat_datalist_filtered_by_year[:10])

### --6.3--Filter All Storms by Month
Filters the file list to return storms active during a specific month across all years (e.g., August). Useful for seasonality analysis.

<i> Similar to filtering by year, here you will filter all the storms by month <br>
 and store these filenames in a list. The filename contains the day of the year that <br> 
 the storm started. This information can be used to filter for storms that <br>
 occurred in a specific month across all years.

The following example filters for all storms that occurred in August (month 08).<br> 
You can filter by any month using the numerical representation for that month.

In [ ]:
'''
6.3 Filter All Storms by Month
This example filters for all storms that occurred in August (month 08).
You can filter by any month using the numerical representation for that month.
After running this cell, use the 'mon_filtered_list' below to run cell 2.5 to
load these data files into a dataframe
'''
mon= 8
mon_filtered_list = filter_by_month(adt_hursat_files_path_list, mon)
# Display the first few elements of the filtered list
print(f"Files filtered by month {mon}:")
display(mon_filtered_list[:10])

### --6.4-- Filter Storms by Year, Month, and Basin
Applies all filters simultaneously to find highly specific data subsets (e.g., "North Atlantic storms in July 2022"). This is the most granular search tool available in the notebook.

<i>You can combine the filters to obtain files for storms that occurred <br>
in a specific basin during a specific month or year or month-year combination.<br>
The key here is to use the basin filtered list from 6.1 named adt_hursat_datalist_filtered_by_basin<br>
if you want to be filtering basin wide data for year and month.<br>
Passing the 'year_basin_filtered_list' variable to the filter_by_month gives you <br>
a single month from a single year from a single basin.  If you were to pass the original <br>
data list used in the filter by basin (6.1) you would get a list similar to that in 6.2 or <br>
6.3, filtered for all basins.<br>

The following example filters for all storms that occurred in the NA basin during July 2022.

In [ ]:
'''
6.4 Filter Storms by Year, Month, and Basin
This example filters for all storms that occurred in the NA basin during July 2022.
The basin filtered list and the year basin filtered lists are provided for convenience.
You can choose the year and month of interest.
After running this cell, use the 'year_basin_filtered_list' or
'mon_year_basin_filtered_list' below to run cell 2.5 to load these data files into dataframe
'''
yyyy = '2022'
year_basin_filtered_list = filter_files_by_year(adt_hursat_datalist_filtered_by_basin, yyyy)
#The above uses the basin filetered list and returns a list of those basin storms for the year given
#That variable is then used to further filter by month below
mon = '08'
mon_year_basin_filtered_list = filter_by_month(year_basin_filtered_list, mon)

print(f"Files filtered by year {yyyy} and month {mon}:")
display(mon_year_basin_filtered_list)

## References For More Information ##
Knapp, K. R., M. C. Kruk, D. H. Levinson, H. J. Diamond, and C. J. Neumann, 2010: The International Best Track Archive for Climate Stewardship (IBTrACS): Unifying tropical cyclone best track data. Bulletin of the American Meteorological Society, 91, 363-376. doi:10.1175/2009BAMS2755.1

J.P. Kossin, K.R. Knapp, T.L. Olander, & C.S. Velden, Global increase in major tropical cyclone exceedance probability over the past four decades, Proc. Natl. Acad. Sci. U.S.A. 117 (22) 11975-11980, https://doi.org/10.1073/pnas.1920849117 (2020).
Olander, T.L. and C.S. Velden, 2019: The Advanced Dvorak Technique (ADT) for estimating
tropical cyclone intensity: update and new capabilities. Weather Forecast, 34, 905-922.

Vecchi, G.A., Landsea, C., Zhang, W. et al. Changes in Atlantic major hurricane frequency since the late-19th century. Nat Commun 12, 4054 (2021). https://doi.org/10.1038/s41467-021-24268-5